# Business entity matching — Amazon ML Challenge 2026 (S1 → S2/S3)

A self-contained, precision-first entity-resolution notebook built from the findings of `dataesplore.ipynb`.
For every **Source-1** business it predicts which **Source-2 / Source-3** records describe the same business.
The decision rule is tuned for **F0.5**, which weights precision twice as much as recall.

**How to use on Kaggle:**
1. Attach the competition dataset (for example `satwiksps/amazon-ml-challenge-2026`).
2. Click **Run All**.
3. The submission is written to `/kaggle/working/submission.tsv`.

The dataset path is detected automatically under `/kaggle/input`. Internet is optional: if `anyascii` can't be
installed, a built-in transliterator is used instead.

| Step | What happens |
|---|---|
| 0 | Install / check dependencies |
| 1 | Write the pipeline package (`entity_matching/`) from the cells below |
| 2 | Sanity-check the hardcoded noise patterns on real examples from the EDA |
| 3 | **Synthetic demo:** 6k-S1 train and 3k-S1 test including the unseen France. Last run: P 0.997, R 0.957, **F0.5 0.989** |
| 4 | **Real competition data:** train with 5-fold OOF, tune F0.5, predict test, write `submission.tsv` |
| 5 | Inspect the OOF scores, feature importance and submission |

**Pipeline:** hardcoded normalisation → country-blocked IDF key blocking → 73 pair/rule/context features →
LightGBM (GroupKFold by S1) → F0.5-tuned threshold + one-owner constraint.
Singletons and doubtful pairs are removed from **training only**; validation scores every held-out S1.

## Step 0 — dependencies

In [1]:
import importlib, os, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
        print(f"ok        {pkg}")
    except ImportError:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           capture_output=True, text=True)
        try:
            importlib.import_module(import_name or pkg)
            print(f"installed {pkg}")
        except ImportError:
            print(f"MISSING   {pkg} (no internet?) " + ("- built-in fallback will be used" if pkg == "anyascii" else ""))

for p in ["pandas", "numpy", "scikit-learn:sklearn", "lightgbm", "rapidfuzz", "anyascii"]:
    name, _, mod = p.partition(":")
    ensure(name, mod or None)

# work in /kaggle/working on Kaggle, the current folder elsewhere
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK)
os.makedirs("entity_matching", exist_ok=True)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
print("working dir:", WORK)

ok        pandas
ok        numpy


ok        scikit-learn
ok        lightgbm
ok        rapidfuzz
ok        anyascii
working dir: /tmp/claude-0/-home-user-my-new/7ea8580c-24a2-5147-be5a-964b18b7aee5/scratchpad/nbexec


## Step 1 — the pipeline package
Each cell writes one module of `entity_matching/`. The modules are kept as real files, not inline code,
so the fork-based worker pools can share memory with the workers.

| Module | Role |
|---|---|
| `patterns.py` | **Hardcoded pattern dictionaries**: legal forms (US/IN/FR), native-script Indic legal words, DBA markers, junk regexes, OCR confusions, US/Indian/French states (incl. native script), street types, city aliases, unit labels |
| `normalize.py` | Name + address normalisers: transliteration, phonetic skeleton, OCR fold, canonical numbers |
| `utils.py` | Kaggle path detection, deadlock-safe fork pool with timeouts, RAM logging |
| `data.py` | TSV loading, parallel normalisation over unique values, cache |
| `blocking.py` | Candidate generation: hashed IDF-weighted keys, country hard block, bounded-RAM chunks |
| `features.py` | 73 pair features, rule flags, context/rank features |
| `metrics.py` | Micro / macro F-beta |
| `synth.py` | Synthetic data with the same noise as the real data (used for the demo) |
| `run.py` | CLI / entry points: `kaggle`, `train`, `predict`, `synth` |

In [2]:
%%writefile entity_matching/__init__.py
"""Business entity matching pipeline (Amazon ML Challenge 2026)."""

Writing entity_matching/__init__.py


In [3]:
%%writefile entity_matching/patterns.py
"""
Hard-coded normalisation patterns for the Amazon ML Challenge 2026
business entity-resolution task (S1 -> S2/S3 matching).

Every dictionary here comes from a noise pattern observed in the EDA
notebook (dataesplore.ipynb, cells 12-20) or from the countries present
in the test set (US, India, and the unseen France).

All keys are lower-case, ASCII-folded, punctuation-free unless noted.
"""

# ---------------------------------------------------------------------------
# Placeholders that mean "no value" (cell 14: null / <null> / none / nan)
# ---------------------------------------------------------------------------
NULL_TOKENS = {
    "null", "<null>", "none", "nan", "n/a", "na", "nil", "unknown",
    "not available", "not applicable", "-", "--", "0",
}

# ---------------------------------------------------------------------------
# Legal / entity forms -> canonical form.
# Multi-word variants are matched on the space-normalised string first.
# ---------------------------------------------------------------------------
LEGAL_MULTIWORD = [
    # order matters: longest first
    ("limited liability partnership", "llp"),
    ("limited liability company", "llc"),
    ("professional limited liability company", "pllc"),
    ("professional corporation", "pc"),
    ("limited partnership", "lp"),
    ("private limited company", "pvt ltd"),
    ("private limited", "pvt ltd"),
    ("public limited company", "public ltd"),
    ("public limited", "public ltd"),
    ("one person company", "opc"),
    ("societe anonyme", "sa"),
    ("societe par actions simplifiee unipersonnelle", "sasu"),
    ("societe par actions simplifiee", "sas"),
    ("societe a responsabilite limitee", "sarl"),
    ("entreprise unipersonnelle a responsabilite limitee", "eurl"),
    ("societe civile immobiliere", "sci"),
    ("et compagnie", "cie"),
    ("et cie", "cie"),
    ("and company", "co"),
    ("and co", "co"),
    ("pvt ltd", "pvt ltd"),
]

LEGAL_TOKENS = {
    # US
    "llc": "llc", "l.l.c": "llc", "llc.": "llc", "lllc": "llc",
    "inc": "inc", "incorporated": "inc", "incorp": "inc", "inco": "inc",
    "corp": "corp", "corporation": "corp", "corpn": "corp",
    "co": "co", "company": "co", "cos": "co",
    "ltd": "ltd", "limited": "ltd", "ltda": "ltd", "lmtd": "ltd", "limted": "ltd",
    "llp": "llp", "lp": "lp", "pllc": "pllc", "pc": "pc", "pa": "pa",
    "plc": "plc", "lc": "lc", "ltd.": "ltd",
    # India
    "pvt": "pvt", "private": "pvt", "pvtltd": "pvt ltd", "prv": "pvt",
    "public": "public", "opc": "opc",
    # France (unseen in train -> hard-coded)
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "sa": "sa", "eurl": "eurl",
    "sci": "sci", "snc": "snc", "scp": "scp", "selarl": "selarl",
    "scop": "scop", "gie": "gie", "cie": "cie", "compagnie": "cie",
    "ets": "ets", "etablissements": "ets", "etablissement": "ets",
    "societe": "societe", "ste": "societe",
    # Generic international
    "gmbh": "gmbh", "ag": "ag", "bv": "bv", "nv": "nv", "srl": "srl",
    "spa": "spa", "oy": "oy", "ab": "ab", "pty": "pty",
}

# Native-script legal words (cells 13/15: Indic-script names keep the legal
# form transliterated phonetically).  Replaced BEFORE transliteration.
INDIC_LEGAL = {
    # Devanagari (Hindi / Marathi)
    "प्राइवेट": " private ", "प्रायवेट": " private ", "प्राईवेट": " private ",
    "लिमिटेड": " limited ", "लिमीटेड": " limited ",
    "प्रा.": " pvt ", "प्रा": " pvt ", "लि.": " ltd ", "लि": " ltd ",
    "एलएलपी": " llp ", "कंपनी": " company ", "कम्पनी": " company ",
    "इंक": " inc ", "कॉर्प": " corp ",
    # Bengali
    "প্রাইভেট": " private ", "লিমিটেড": " limited ", "প্রা.": " pvt ",
    "লি.": " ltd ", "এলএলপি": " llp ",
    # Tamil
    "பிரைவேட்": " private ", "லிமிடெட்": " limited ", "எல்எல்பி": " llp ",
    # Telugu
    "ప్రైవేట్": " private ", "లిమిటెడ్": " limited ", "ఎల్ఎల్పి": " llp ",
    # Kannada
    "ಪ್ರೈವೇಟ್": " private ", "ಲಿಮಿಟೆಡ್": " limited ", "ಎಲ್ಎಲ್ಪಿ": " llp ",
    # Gujarati
    "પ્રાઇવેટ": " private ", "લિમિટેડ": " limited ", "પ્રા.": " pvt ",
    "લિ.": " ltd ", "એલએલપી": " llp ",
    # Malayalam
    "പ്രൈവറ്റ്": " private ", "ലിമിറ്റഡ്": " limited ", "എൽഎൽപി": " llp ",
    # Punjabi (Gurmukhi) / Odia – rarer but cheap to include
    "ਪ੍ਰਾਈਵੇਟ": " private ", "ਲਿਮਿਟੇਡ": " limited ",
    "ପ୍ରାଇଭେଟ": " private ", "ଲିମିଟେଡ": " limited ",
}

# Phonetic skeletons of legal words: catches transliterations such as
# "praivet", "piraivet", "limitet", "limitedd" (see normalize.skeleton()).
LEGAL_SKELETON_WORDS = [
    "private", "limited", "pvt", "ltd", "llp", "company", "incorporated",
    "corporation", "praivet", "piraivet", "limitet", "elelpi",
]

# DBA / alias markers (cells 15-16): "X D.B.A. Y", "Korzephdova formerly: Y",
# "Viovera | www.viovera.com".
ALIAS_MARKERS = [
    r"\bd\s*/\s*b\s*/\s*a\b", r"\bd\.?\s?b\.?\s?a\.?\b", r"\bdba\b",
    r"\bt\s*/\s*a\b", r"\btrading\s+as\b", r"\bdoing\s+business\s+as\b",
    r"\bformerly\s+known\s+as\b", r"\bformerly\b", r"\bf\s*/\s*k\s*/\s*a\b",
    r"\bfka\b", r"\ba\s*/\s*k\s*/\s*a\b", r"\baka\b", r"\balso\s+known\s+as\b",
    r"\bsucc(?:essor)?\s+to\b", r"\bnee\b", r"\|",
]

# Junk that the noisy sources prepend/append (cells 13-15).
JUNK_REGEXES = [
    r"\(\s*id\s*[:#]?\s*\d+\s*\)",      # (ID: 49166)
    r"\bid\s*[:#]\s*\d+\b",              # ID: 49166
    r"#\s*\d{3,}\b",                     # #98587
    r"^\W+",                             # ***, --, >>, # at start
    r"\W+$",                             # trailing symbols
    r"\bm\s*/\s*s\b\.?",                  # M/s  (Indian "Messrs"), anywhere
    r"^\s*messrs\.?\s+",
    r"\bwww\.",                          # www.
    r"https?://",
]

# Words that carry no identity in a business name.
NAME_STOPWORDS = {
    "the", "of", "and", "a", "an", "for", "at", "in", "on", "by",
    # French
    "de", "la", "le", "les", "du", "des", "et", "l", "d", "au", "aux",
    # Indian honorifics that noisy sources add/drop ("Sri", "Shri", "M/s")
    "m/s", "ms", "messrs",
}
HONORIFICS = {"sri", "shri", "shree", "smt", "dr", "mr", "mrs", "st"}

# Top-level domains stripped from names written as URLs ("aahanarealty.com").
TLDS = {
    "com", "net", "org", "in", "co", "io", "biz", "info", "us", "fr",
    "co.in", "org.in", "net.in", "edu", "gov", "c0m", "corn",
}

# ---------------------------------------------------------------------------
# OCR / "leet" confusions inside alphabetic tokens (cell 15/16: 5outh,
# WASHINGT0N, lnc, FAMlLY, silvergrill.c0m, Dermato1ogy, MEMORlAL).
# Applied symmetrically to BOTH sides to build an OCR-folded comparison key.
# ---------------------------------------------------------------------------
OCR_DIGIT_TO_CHAR = {
    "0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "6": "g", "7": "t",
    "8": "b", "9": "g", "@": "a", "$": "s", "|": "l", "!": "l",
}
# After digit fixing, collapse visually confusable letters.
OCR_CHAR_FOLD = [("rn", "m"), ("vv", "w"), ("cl", "d"), ("l", "i"), ("j", "i")]

# ---------------------------------------------------------------------------
# Address: US states (name <-> USPS code)
# ---------------------------------------------------------------------------
US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar",
    "california": "ca", "colorado": "co", "connecticut": "ct", "delaware": "de",
    "district of columbia": "dc", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in",
    "iowa": "ia", "kansas": "ks", "kentucky": "ky", "louisiana": "la",
    "maine": "me", "maryland": "md", "massachusetts": "ma", "michigan": "mi",
    "minnesota": "mn", "mississippi": "ms", "missouri": "mo", "montana": "mt",
    "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj",
    "new mexico": "nm", "new york": "ny", "north carolina": "nc",
    "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or",
    "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut",
    "vermont": "vt", "virginia": "va", "washington": "wa",
    "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "puerto rico": "pr", "guam": "gu", "virgin islands": "vi",
}

# ---------------------------------------------------------------------------
# Address: Indian states / UTs (name, common code, native-script spelling)
# ---------------------------------------------------------------------------
INDIA_STATES = {
    "andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as",
    "bihar": "br", "chhattisgarh": "cg", "chattisgarh": "cg", "goa": "ga",
    "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp",
    "jharkhand": "jh", "karnataka": "ka", "kerala": "kl", "keralam": "kl",
    "madhya pradesh": "mp", "maharashtra": "mh", "manipur": "mn",
    "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl", "odisha": "od",
    "orissa": "od", "punjab": "pb", "rajasthan": "rj", "sikkim": "sk",
    "tamil nadu": "tn", "tamilnadu": "tn", "telangana": "tg", "tripura": "tr",
    "uttar pradesh": "up", "uttarakhand": "uk", "uttaranchal": "uk",
    "west bengal": "wb", "delhi": "dl", "new delhi": "dl", "nct of delhi": "dl",
    "jammu and kashmir": "jk", "jammu kashmir": "jk", "ladakh": "la",
    "chandigarh": "ch", "puducherry": "py", "pondicherry": "py",
    "andaman and nicobar islands": "an", "lakshadweep": "ld",
    "dadra and nagar haveli": "dn", "daman and diu": "dd",
}
INDIA_STATE_CODE_ALIASES = {"ts": "tg", "or": "od", "ua": "uk", "ct": "cg"}

# Native-script state names seen in S2/S3 addresses (cell 13).
INDIA_STATES_NATIVE = {
    "महाराष्ट्र": "mh", "दिल्ली": "dl", "राजस्थान": "rj", "मध्य प्रदेश": "mp",
    "उत्तर प्रदेश": "up", "बिहार": "br", "हरियाणा": "hr", "पंजाब": "pb",
    "उत्तराखंड": "uk", "झारखंड": "jh", "छत्तीसगढ़": "cg", "हिमाचल प्रदेश": "hp",
    "गोवा": "ga", "ಕರ್ನಾಟಕ": "ka", "தமிழ்நாடு": "tn", "తెలంగాణ": "tg",
    "ఆంధ్ర ప్రదేశ్": "ap", "ఆంధ్రప్రదేశ్": "ap", "পশ্চিমবঙ্গ": "wb",
    "ગુજરાત": "gj", "കേരളം": "kl", "ଓଡ଼ିଶା": "od", "ਪੰਜਾਬ": "pb",
    "অসম": "as", "आसाम": "as",
}

# ---------------------------------------------------------------------------
# France regions / departments present in test (cell 12, descriptive only)
# ---------------------------------------------------------------------------
FRANCE_REGIONS = {
    "hauts de france": "hdf", "nouvelle aquitaine": "naq",
    "pays de la loire": "pdl", "ile de france": "idf", "bretagne": "bre",
    "normandie": "nor", "grand est": "ges", "occitanie": "occ",
    "auvergne rhone alpes": "ara", "provence alpes cote d azur": "pac",
    "bourgogne franche comte": "bfc", "centre val de loire": "cvl",
    "corse": "cor", "nord": "d59", "pas de calais": "d62", "gironde": "d33",
    "loire atlantique": "d44",
}

# ---------------------------------------------------------------------------
# City aliases (old/new names, common misspellings)
# ---------------------------------------------------------------------------
CITY_ALIASES = {
    "bengaluru": "bangalore", "bangaluru": "bangalore", "bombay": "mumbai",
    "madras": "chennai", "calcutta": "kolkata", "gurugram": "gurgaon",
    "thiruvananthapuram": "trivandrum", "poona": "pune", "baroda": "vadodara",
    "prayagraj": "allahabad", "vizag": "visakhapatnam",
    "vishakhapatnam": "visakhapatnam", "vishakapatnam": "visakhapatnam",
    "mysuru": "mysore", "mangaluru": "mangalore", "cochin": "kochi",
    "belagavi": "belgaum", "kalaburagi": "gulbarga", "cawnpore": "kanpur",
    "benares": "varanasi", "banaras": "varanasi", "simla": "shimla",
    "ahmedbad": "ahmedabad", "amdavad": "ahmedabad", "secunderabad": "hyderabad",
    "hubballi": "hubli", "tiruchirappalli": "trichy", "tiruchirapalli": "trichy",
    "puducherry": "pondicherry", "nw delhi": "new delhi", "navi mumbai": "navi mumbai",
    "gautam budh nagar": "gautam buddha nagar", "goutam budd nagar": "gautam buddha nagar",
    "noida": "gautam buddha nagar",
}

# ---------------------------------------------------------------------------
# Street / thoroughfare types (USPS Pub 28 + Indian + French) -> canonical
# ---------------------------------------------------------------------------
STREET_TYPES = {
    # US
    "street": "st", "str": "st", "st": "st", "saint": "st",  # "25 PALMER SAINT"
    "road": "rd", "rd": "rd", "avenue": "ave", "ave": "ave", "av": "ave",
    "aven": "ave", "avn": "ave", "drive": "dr", "dr": "dr", "drv": "dr",
    "lane": "ln", "ln": "ln", "court": "ct", "ct": "ct", "crt": "ct",
    "boulevard": "blvd", "blvd": "blvd", "boul": "blvd", "bd": "blvd",
    "place": "pl", "pl": "pl", "terrace": "ter", "ter": "ter", "terr": "ter",
    "trail": "trl", "trl": "trl", "circle": "cir", "cir": "cir", "circ": "cir",
    "parkway": "pkwy", "pkwy": "pkwy", "pky": "pkwy", "highway": "hwy",
    "hwy": "hwy", "square": "sq", "sq": "sq", "way": "way", "wy": "way",
    "expressway": "expy", "expy": "expy", "freeway": "fwy", "fwy": "fwy",
    "turnpike": "tpke", "tpke": "tpke", "crossing": "xing", "xing": "xing",
    "point": "pt", "pt": "pt", "pointe": "pt", "heights": "hts", "hts": "hts",
    "mount": "mt", "mt": "mt", "mountain": "mtn", "mtn": "mtn",
    "center": "ctr", "centre": "ctr", "ctr": "ctr", "plaza": "plz", "plz": "plz",
    "alley": "aly", "aly": "aly", "loop": "loop", "run": "run", "row": "row",
    "pike": "pike", "path": "path", "walk": "walk", "cove": "cv", "cv": "cv",
    "creek": "crk", "crk": "crk", "ridge": "rdg", "rdg": "rdg",
    "hollow": "holw", "holw": "holw", "valley": "vly", "vly": "vly",
    "junction": "jct", "jct": "jct", "estates": "est", "est": "est",
    "station": "sta", "sta": "sta", "township": "twp", "twp": "twp",
    "townhsip": "twp",
    # India
    "marg": "marg", "mg": "marg", "nagar": "nagar", "ngr": "nagar",
    "colony": "colony", "col": "colony", "sector": "sector", "sec": "sector",
    "phase": "phase", "ph": "phase", "block": "block", "blk": "block",
    "layout": "layout", "extension": "extn", "extn": "extn", "ext": "extn",
    "chowk": "chowk", "gali": "gali", "mohalla": "mohalla", "bazar": "bazaar",
    "bazaar": "bazaar", "cross": "cross", "main": "main", "salai": "salai",
    "industrial": "indl", "indl": "indl", "ind": "indl", "estate": "est",
    "complex": "cplx", "cplx": "cplx", "building": "bldg", "bldg": "bldg",
    "bldng": "bldg", "tower": "twr", "twr": "twr", "apartment": "apt",
    "apartments": "apt", "apts": "apt", "apt": "apt", "society": "soc",
    "soc": "soc", "chs": "soc", "near": "nr", "nr": "nr", "opposite": "opp",
    "opp": "opp", "behind": "bhd", "bh": "bhd", "post": "po", "po": "po",
    "village": "vill", "vill": "vill", "vil": "vill", "taluka": "tal",
    "tal": "tal", "tehsil": "tal", "district": "dist", "dist": "dist",
    "distt": "dist", "railway": "rly", "rly": "rly",
    # France
    "rue": "rue", "r": "rue", "allee": "all", "all": "all", "impasse": "imp",
    "imp": "imp", "chemin": "che", "che": "che", "ch": "che", "route": "rte",
    "rte": "rte", "quai": "quai", "cours": "crs", "crs": "crs", "residence": "res",
    "res": "res", "lieu dit": "ld", "lieudit": "ld", "ld": "ld", "faubourg": "fbg",
    "fbg": "fbg", "passage": "pass", "pass": "pass", "sentier": "sen",
    "hameau": "ham", "ham": "ham", "zone": "zone", "za": "za", "zi": "zi",
    "zac": "zac", "bis": "bis", "ter.": "ter", "sainte": "ste", "ste": "ste",
    "cedex": "",
}

DIRECTIONALS = {
    "north": "n", "south": "s", "east": "e", "west": "w", "northeast": "ne",
    "northwest": "nw", "southeast": "se", "southwest": "sw", "n": "n",
    "s": "s", "e": "e", "w": "w", "ne": "ne", "nw": "nw", "se": "se",
    "sw": "sw", "nord": "n", "sud": "s", "est": "e", "ouest": "w",
}

# Unit / house labels that precede a number (cell 14): the label itself is
# dropped, the number is kept.
UNIT_LABELS = {
    "unit", "apt", "apartment", "suite", "ste", "room", "rm", "floor", "fl",
    "flr", "office", "off", "shop", "plot", "flat", "house", "door", "no",
    "number", "num", "nos", "hno", "h.no", "hn", "dno", "d.no", "fno", "f.no",
    "khasra", "kh", "survey", "sr", "sy", "s.no", "sno", "ward", "wing",
    "bldg", "building", "block", "pmb", "box", "po box", "pobox", "#",
    "premises", "cabin", "gala", "stall", "site", "khata", "cts", "rs",
    "gat", "gut", "pl", "bat", "batiment", "etage", "appt", "porte",
}

ORDINAL_WORDS = {
    "first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5",
    "sixth": "6", "seventh": "7", "eighth": "8", "ninth": "9", "tenth": "10",
    "eleventh": "11", "twelfth": "12", "ground": "0", "gf": "0",
    "premier": "1", "deuxieme": "2", "troisieme": "3", "rdc": "0",
}

ADDRESS_STOPWORDS = {
    "of", "the", "and", "at", "in", "near", "city", "town", "de", "la", "le",
    "du", "des", "les", "l", "d", "s/o", "c/o", "w/o", "d/o", "county",
    "region", "india", "usa", "us", "france", "united", "states",
}

Writing entity_matching/patterns.py


In [4]:
%%writefile entity_matching/normalize.py
"""
Name / address normalisation.

Every function is pure (str -> dict of str) so it can be mapped over the
unique values of a column with multiprocessing.
"""
from __future__ import annotations

import re
import unicodedata
from functools import lru_cache

from . import patterns as P

try:  # best transliterator for all Indic scripts + accent folding
    from anyascii import anyascii as _anyascii
except ImportError:  # pragma: no cover - fallback keeps the pipeline running
    _anyascii = None

# ---------------------------------------------------------------------------
# Unicode helpers
# ---------------------------------------------------------------------------
_ZERO_WIDTH = re.compile(r"[​-‍﻿­]")
INDIC_RE = re.compile(r"[ऀ-෿]")
_NON_ASCII = re.compile(r"[^\x00-\x7F]")
_WS = re.compile(r"\s+")

_INDIC_LEGAL_SORTED = sorted(P.INDIC_LEGAL.items(), key=lambda kv: -len(kv[0]))
_NATIVE_STATES_SORTED = sorted(P.INDIA_STATES_NATIVE.items(), key=lambda kv: -len(kv[0]))


def _fallback_translit(text: str) -> str:
    """Transliterate without anyascii: accents via NFKD, Brahmic scripts via
    Unicode character names ("DEVANAGARI LETTER RA" -> "ra")."""
    out = []
    for ch in unicodedata.normalize("NFKD", text):
        if ord(ch) < 128:
            out.append(ch)
            continue
        if unicodedata.combining(ch) and not INDIC_RE.match(ch):
            continue
        name = unicodedata.name(ch, "")
        if " LETTER " in name:
            syl = name.split(" LETTER ")[-1].lower().split()[0]
            out.append(syl if len(syl) <= 3 else syl[:3])
        elif " VOWEL SIGN " in name:
            out.append(name.split(" VOWEL SIGN ")[-1].lower().split()[0][:2])
        elif " DIGIT " in name:
            out.append(str(unicodedata.digit(ch, 0)))
        elif "VIRAMA" in name and out and out[-1].endswith("a"):
            out[-1] = out[-1][:-1]
        else:
            out.append(" ")
    return "".join(out)


def to_ascii(text: str) -> str:
    if not _NON_ASCII.search(text):
        return text
    if _anyascii is not None:
        return _anyascii(text)
    return _fallback_translit(text)


def clean_unicode(text) -> str:
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = _ZERO_WIDTH.sub("", text)
    return text.strip()


# ---------------------------------------------------------------------------
# Phonetic skeleton - makes transliterated Indic names comparable with the
# English spelling ("sliusns" ~ "solutions", "piraivet" ~ "private").
# ---------------------------------------------------------------------------
_SKEL_SUBS = [
    ("tion", "sn"), ("sion", "sn"), ("tian", "sn"), ("ph", "f"), ("sh", "s"),
    ("ch", "c"), ("th", "t"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
    ("dh", "d"), ("jh", "j"), ("ck", "k"), ("qu", "k"), ("x", "ks"),
    ("w", "v"), ("z", "s"), ("ce", "se"), ("ci", "si"), ("cy", "sy"),
    ("ge", "je"), ("gi", "ji"), ("gy", "jy"),
    ("c", "k"), ("q", "k"), ("b", "p"), ("d", "t"),
    ("g", "k"), ("m", "n"),
]
_VOWELS = set("aeiouyh")


@lru_cache(maxsize=500_000)
def skeleton(token: str) -> str:
    if not token:
        return ""
    if token.isdigit():
        return token.lstrip("0") or "0"
    t = token
    for a, b in _SKEL_SUBS:
        t = t.replace(a, b)
    head, rest = t[0], [c for c in t[1:] if c not in _VOWELS]
    out = [head]
    for c in rest:
        if c != out[-1]:
            out.append(c)
    return "".join(out)


_LEGAL_SKELETONS = {skeleton(w) for w in P.LEGAL_SKELETON_WORDS}


def ocr_fold(text: str) -> str:
    """Symmetric OCR folding for comparison keys only."""
    out = []
    for tok in text.split():
        if any(c.isalpha() for c in tok):
            tok = "".join(P.OCR_DIGIT_TO_CHAR.get(c, c) for c in tok)
        for a, b in P.OCR_CHAR_FOLD:
            tok = tok.replace(a, b)
        out.append(tok)
    return " ".join(out)


# ---------------------------------------------------------------------------
# Business names
# ---------------------------------------------------------------------------
_ALIAS_RE = re.compile("|".join(P.ALIAS_MARKERS))
_JUNK_RES = [re.compile(r) for r in P.JUNK_REGEXES]
_DOTTED_ABBR = re.compile(r"\b(?:[a-z]\.){2,}[a-z]?\.?")
_DOMAIN_RE = re.compile(
    r"^(?:www\.)?([a-z0-9][a-z0-9\-]*?)\.(" + "|".join(
        re.escape(t) for t in sorted(P.TLDS, key=len, reverse=True)) + r")\.?$")
_NAME_PUNCT = re.compile(r"[^a-z0-9\s]")
_LEGAL_MULTI = [(re.compile(r"\b" + re.escape(a) + r"\b"), b) for a, b in P.LEGAL_MULTIWORD]
_LEGAL_TOKENS = dict(P.LEGAL_TOKENS, lnc="inc", ilc="llc", iic="llc", ltdd="ltd")


def _fix_ocr_token(tok: str) -> str:
    # digits inside a word are OCR noise: 5outh, washingt0n, dermato1ogy
    if any(c.isdigit() for c in tok) and sum(c.isalpha() for c in tok) >= 2:
        return "".join(P.OCR_DIGIT_TO_CHAR.get(c, c) for c in tok)
    return tok


def _clean_single_name(s: str, from_indic: bool):
    flags = {"domain": 0, "junk": 0}
    s = s.strip()
    for rx in _JUNK_RES:
        new = rx.sub(" ", s)
        if new != s:
            flags["junk"] = 1
            s = new
    s = s.strip()
    # whole name written as a web domain: aahanarealty.com / silvergrill.c0m
    m = _DOMAIN_RE.match(s.replace(" ", ""))
    if m and " " not in s.strip():
        s = m.group(1).replace("-", " ")
        flags["domain"] = 1
    s = _DOTTED_ABBR.sub(lambda mm: mm.group(0).replace(".", ""), s)
    s = s.replace("&", " and ").replace("+", " and ").replace("'s ", "s ")
    s = s.replace("'", "")
    s = _NAME_PUNCT.sub(" ", s)
    s = _WS.sub(" ", s).strip()
    for rx, rep in _LEGAL_MULTI:
        s = rx.sub(rep, s)

    core, legal = [], set()
    for tok in s.split():
        tok = _fix_ocr_token(tok)
        if tok in _LEGAL_TOKENS:
            for lg in _LEGAL_TOKENS[tok].split():
                legal.add(lg)
            continue
        if from_indic and len(tok) > 2 and skeleton(tok) in _LEGAL_SKELETONS:
            legal.add(tok)
            continue
        if tok in P.NAME_STOPWORDS or tok in P.HONORIFICS:
            continue
        if core and core[-1] == tok:  # repeated word: "bright bright"
            continue
        core.append(tok)
    if not core and legal:  # name was only legal words - keep something
        core = sorted(legal)
    return core, legal, flags


@lru_cache(maxsize=None)
def normalize_name(raw) -> dict:
    s = clean_unicode(raw)
    has_indic = bool(INDIC_RE.search(s))
    if has_indic:
        for native, rep in _INDIC_LEGAL_SORTED:
            s = s.replace(native, rep)
    s = to_ascii(s).lower()

    parts = [p for p in _ALIAS_RE.split(s) if p and p.strip()]
    if not parts:
        parts = [s]
    alts = []
    legal_all: set = set()
    flags = {"domain": 0, "junk": 0}
    for p in parts:
        core, legal, fl = _clean_single_name(p, has_indic)
        if core:
            alts.append(core)
        legal_all |= legal
        for k in flags:
            flags[k] = max(flags[k], fl[k])
    if not alts:
        alts = [[]]
    # primary = the longest alternative (usually the real business name)
    main = max(alts, key=lambda c: len(" ".join(c)))
    core_str = " ".join(main)
    skel = " ".join(skeleton(t) for t in main)
    return {
        "core": core_str,
        "full": (core_str + " " + " ".join(sorted(legal_all))).strip(),
        "ocr": ocr_fold(core_str),
        "skel": skel,
        "nospace": core_str.replace(" ", ""),
        "alts": "||".join(dict.fromkeys(" ".join(a) for a in alts if a)) if len(alts) > 1 else "",
        "legal": " ".join(sorted(legal_all)),
        "acr": "".join(t[0] for t in main) if len(main) >= 2 else "",
        "indic": int(has_indic),
        "domain": flags["domain"],
        "junk": int(flags["junk"] or len(alts) > 1),
    }


# ---------------------------------------------------------------------------
# Addresses
# ---------------------------------------------------------------------------
_NULL_RE = re.compile(r"<\s*null\s*>|\b(?:null|none|nan|n/a)\b", re.I)
_ADDR_PUNCT = re.compile(r"[^a-z0-9\s/\-#,]")
_ORD_SUFFIX = re.compile(r"^(\d+)(st|nd|rd|th|er|e|eme|re)$")
_DIGITS = re.compile(r"\d+")
_LABEL_GLUE = re.compile(r"\b(h|d|f|s|sy|kh|r|p)\s*\.\s*no(?=\b|\d)")  # h.no -> hno
_LABEL_DIGIT = re.compile(r"\b(no|hno|dno|fno|sno|plot|flat|door|shop|unit|apt|suite|ste|room|office|house)(?=\d)")
_WORD_HYPHEN = re.compile(r"(?<=[a-z])-(?=[a-z])")
_FRANCE_ONLY = {"r", "ch", "all", "che", "imp", "ld", "res"}
_NO_GLUE = re.compile(r"\bno\s*[.:]\s*")
_HASH = re.compile(r"#+\s*")

_US_CODES = set(P.US_STATES.values())
_IN_CODES = set(P.INDIA_STATES.values()) | set(P.INDIA_STATE_CODE_ALIASES)
_ALL_STATE_NAMES = sorted(
    list(P.US_STATES.items()) + list(P.INDIA_STATES.items()) + list(P.FRANCE_REGIONS.items()),
    key=lambda kv: -len(kv[0]))
_STATE_NAME_RES = [(re.compile(r"\b" + re.escape(n) + r"\b"), c) for n, c in _ALL_STATE_NAMES]
_CITY_ALIAS_RES = [(re.compile(r"\b" + re.escape(a) + r"\b"), c)
                   for a, c in sorted(P.CITY_ALIASES.items(), key=lambda kv: -len(kv[0]))]
_STATE_CODES = _US_CODES | _IN_CODES | set(P.FRANCE_REGIONS.values())


def canon_number(tok: str) -> str:
    """0271a -> 271a, 4Nd -> 4, 05131 -> 5131, b-204 -> b-204."""
    m = _ORD_SUFFIX.match(tok)
    if m:
        return m.group(1).lstrip("0") or "0"
    return _DIGITS.sub(lambda mm: mm.group(0).lstrip("0") or "0", tok).strip("-/")


@lru_cache(maxsize=None)
def normalize_address(raw, country: str = "") -> dict:
    s = clean_unicode(raw)
    for native, code in _NATIVE_STATES_SORTED:
        if native in s:
            s = s.replace(native, f" {code} ")
    s = to_ascii(s).lower()
    s = _NULL_RE.sub(" ", s)
    s = _LABEL_GLUE.sub(lambda m: m.group(1) + "no ", s)
    s = _NO_GLUE.sub("no ", s)
    s = _LABEL_DIGIT.sub(lambda m: m.group(1) + " ", s)
    s = _WORD_HYPHEN.sub(" ", s)
    s = _HASH.sub(" # ", s)
    s = s.replace(".", " ").replace(";", ",").replace("(", " ").replace(")", " ")
    s = _ADDR_PUNCT.sub(" ", s)
    for rx, c in _CITY_ALIAS_RES:
        s = rx.sub(c, s)

    alpha, nums, numparts, numstreet, states = [], [], set(), [], set()
    ordered = []
    for comp in s.split(","):
        comp = _WS.sub(" ", comp).strip(" -/")
        if not comp:
            continue
        if comp in _STATE_CODES and (
                (country == "US" and comp in _US_CODES)
                or (country == "India" and (comp in _IN_CODES))
                or country not in ("US", "India")):
            states.add(P.INDIA_STATE_CODE_ALIASES.get(comp, comp) if country == "India" else comp)
            continue
        for rx, c in _STATE_NAME_RES:
            if rx.search(comp):
                states.add(c)
                comp = rx.sub(" ", comp)
        toks = comp.split()
        prev_num = None
        for i, tok in enumerate(toks):
            tok = tok.strip("-/")
            if not tok or tok == "#":
                continue
            if tok in P.ORDINAL_WORDS and i + 1 < len(toks) and toks[i + 1] in ("floor", "fl", "flr", "etage"):
                tok = P.ORDINAL_WORDS[tok]
            if any(c.isdigit() for c in tok):
                cn = canon_number(tok)
                if cn:
                    nums.append(cn)
                    numparts.update(d.lstrip("0") or "0" for d in _DIGITS.findall(tok))
                    ordered.append(cn)
                    prev_num = cn
                continue
            if tok in P.UNIT_LABELS or tok in P.ADDRESS_STOPWORDS:
                continue
            if country == "France" or tok not in _FRANCE_ONLY:
                tok = P.STREET_TYPES.get(tok, tok)
            tok = P.DIRECTIONALS.get(tok, tok)
            if not tok:
                continue
            if len(tok) == 2 and tok in _STATE_CODES and i == len(toks) - 1 and len(toks) <= 2:
                states.add(tok)
                continue
            alpha.append(tok)
            ordered.append(tok)
            if prev_num is not None:
                numstreet.append(f"{prev_num}_{tok}")
                prev_num = None

    postal = sorted(n for n in nums if n.isdigit() and len(n) in (5, 6))
    return {
        "atext": " ".join(ordered),
        "alpha": " ".join(alpha),
        "nums": " ".join(nums),
        "numparts": " ".join(sorted(numparts)),
        "numstreet": " ".join(numstreet),
        "first_num": nums[0] if nums else "",
        "state": " ".join(sorted(states)),
        "postal": " ".join(postal),
        "missing": int(not ordered),
    }

Writing entity_matching/normalize.py


In [5]:
%%writefile entity_matching/utils.py
"""Kaggle helpers: path auto-detection, deadlock-safe process pools, memory logs."""
from __future__ import annotations

import gc
import os
import sys
import time
from multiprocessing import get_context
from multiprocessing.context import TimeoutError as MPTimeout
from pathlib import Path

_T0 = time.time()

# Keep native thread pools single-threaded inside forked workers: forking a
# process whose OpenMP/BLAS pool is live and then using that pool in the
# child is the classic multiprocessing deadlock.  Set before numpy/lightgbm
# spin their pools up in the children.
_THREAD_VARS = ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS")

REQUIRED = ("train_source1.tsv", "train_source2.tsv", "train_source3.tsv")
SEARCH_ROOTS = ("/kaggle/input", "/kaggle/working", ".", "./dataset", "./data", "..")


def rss_gb() -> float:
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024 ** 2
    except OSError:
        pass
    return float("nan")


def log(msg: str) -> None:
    print(f"[{time.time() - _T0:7.1f}s | {rss_gb():5.1f} GB] {msg}", flush=True)


def free() -> None:
    gc.collect()


def is_kaggle() -> bool:
    return Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def auto_data_dir(explicit: str | None = None, split: str = "train") -> Path:
    """Find the folder tree holding the challenge TSVs.

    Checks the explicit path, $EM_DATA_DIR, then searches /kaggle/input (any
    dataset/competition slug, any nesting depth) and local folders for
    `{split}_source1.tsv`.  Returns the common root that contains both the
    train and test files when possible."""
    cands = [explicit, os.environ.get("EM_DATA_DIR")] + list(SEARCH_ROOTS)
    for root in cands:
        if not root:
            continue
        root = Path(root)
        if not root.is_dir():
            continue
        hits = sorted(root.rglob(f"{split}_source1.tsv"))
        if not hits:
            continue
        # prefer a root that also contains the other split
        for h in hits:
            for parent in [h.parent, h.parent.parent]:
                names = {p.name for p in parent.rglob("*.tsv")}
                if {"train_source1.tsv", "test_source1.tsv"} <= names:
                    return parent
        return hits[0].parent.parent if hits[0].parent.name == split else hits[0].parent
    raise FileNotFoundError(
        f"Could not find {split}_source1.tsv under {', '.join(str(c) for c in cands if c)}. "
        "Attach the dataset to the notebook or pass --data-dir.")


def default_out_dir() -> Path:
    return Path("/kaggle/working/em_model") if is_kaggle() else Path("em_output")


def n_workers(requested: int | None = None) -> int:
    if requested:
        return max(1, int(requested))
    if hasattr(os, "sched_getaffinity"):
        return max(1, len(os.sched_getaffinity(0)))
    return max(1, os.cpu_count() or 1)


def _init_worker():
    for v in _THREAD_VARS:
        os.environ[v] = "1"
    try:  # workers must not handle Ctrl-C / kernel interrupts themselves
        import signal
        signal.signal(signal.SIGINT, signal.SIG_IGN)
    except Exception:
        pass


def run_pool(fn, tasks, n_jobs: int, ordered: bool = True, timeout: float = 3600.0):
    """Map `fn` over `tasks` with a fork pool.

    - fork shares the big read-only arrays with workers (no pickling, no copy)
    - every result has a timeout: a stuck/killed worker (e.g. OOM-killed by
      the kernel) can no longer hang the notebook forever; remaining tasks are
      finished serially instead
    - falls back to serial execution on platforms without fork"""
    tasks = list(tasks)
    if n_jobs <= 1 or len(tasks) <= 1 or sys.platform == "win32":
        return [fn(t) for t in tasks]
    results = [None] * len(tasks)
    done = [False] * len(tasks)
    ctx = get_context("fork")
    pool = ctx.Pool(min(n_jobs, len(tasks)), initializer=_init_worker)
    try:
        asyncs = [pool.apply_async(fn, (t,)) for t in tasks]
        for i, ar in enumerate(asyncs):
            try:
                results[i] = ar.get(timeout=timeout)
                done[i] = True
            except MPTimeout:
                log(f"  worker timeout on task {i}; finishing remaining tasks serially")
                break
        pool.close()
    except BaseException:
        pool.terminate()
        raise
    finally:
        if not all(done):
            pool.terminate()
        pool.join()
    for i, ok in enumerate(done):
        if not ok:
            results[i] = fn(tasks[i])
    return results

Writing entity_matching/utils.py


In [6]:
%%writefile entity_matching/data.py
"""Loading the challenge TSVs and attaching normalised columns."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .normalize import normalize_address, normalize_name
from .utils import free, log as _log, n_workers, run_pool

USECOLS = ["entity_id", "business_name", "business_address", "country"]
_G: dict = {}

NAME_FIELDS = ["core", "full", "ocr", "skel", "nospace", "alts", "legal", "acr",
               "indic", "domain", "junk"]
ADDR_FIELDS = ["atext", "alpha", "nums", "numparts", "numstreet", "first_num",
               "state", "postal", "missing"]


def find_files(data_dir: str | Path) -> dict:
    found = {p.name: p for p in Path(data_dir).rglob("*.tsv")}
    return found


def read_tsv(path, usecols=None) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False,
                       usecols=usecols, engine="c", na_filter=False)


def load_split(data_dir, split: str):
    """Returns (s1, targets, ground_truth or None). targets = S2 + S3."""
    f = find_files(data_dir)
    s1 = read_tsv(f[f"{split}_source1.tsv"], USECOLS)
    s2 = read_tsv(f[f"{split}_source2.tsv"], USECOLS)
    s3 = read_tsv(f[f"{split}_source3.tsv"], USECOLS)
    s2["source"] = "S2"
    s3["source"] = "S3"
    tgt = pd.concat([s2, s3], ignore_index=True)
    del s2, s3
    free()
    gt = None
    if f"{split}_ground_truth.tsv" in f:
        gt = read_tsv(f[f"{split}_ground_truth.tsv"])
    for df in (s1, tgt):
        df["country"] = df["country"].astype(str).str.strip().astype("category")
    tgt["source"] = tgt["source"].astype("category")
    return s1, tgt, gt


def gt_pairs(gt: pd.DataFrame) -> pd.DataFrame:
    """Explode 'S2-1,S3-2' -> one row per (source1_entity_id, matched_id)."""
    g = gt[["source1_entity_id", "matched_entity_ids"]].copy()
    g["matched_id"] = g["matched_entity_ids"].astype(str).str.split(",")
    g = g.explode("matched_id")
    g["matched_id"] = g["matched_id"].astype(str).str.strip()
    g = g[g["matched_id"].ne("") & g["matched_id"].ne("nan")]
    return g[["source1_entity_id", "matched_id"]].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Parallel normalisation over unique values (fork-shared input, no pickling)
# ---------------------------------------------------------------------------
def _name_worker(b):
    vals = _G["vals"]
    return [tuple(normalize_name(v)[k] for k in NAME_FIELDS) for v in vals[b[0]:b[1]]]


def _addr_worker(b):
    vals, ctry = _G["vals"], _G["ctry"]
    return [tuple(normalize_address(v, c)[k] for k in ADDR_FIELDS)
            for v, c in zip(vals[b[0]:b[1]], ctry[b[0]:b[1]])]


def _run(fn, n, n_jobs, chunk=20000):
    out = []
    for res in run_pool(fn, [(i, min(i + chunk, n)) for i in range(0, n, chunk)], n_jobs):
        out.extend(res)
    return out


def _assign(df, prefix, fields, int_fields, res, codes):
    for j, k in enumerate(fields):
        col = np.array([r[j] for r in res], dtype=object)[codes]
        df[prefix + k] = col.astype(np.int8) if k in int_fields else col


def normalize_frame(df: pd.DataFrame, n_jobs: int | None = None, log=_log,
                    drop_raw: bool = True) -> pd.DataFrame:
    """Adds n_* (name) and a_* (address) columns.  Work is done once per
    UNIQUE value; the resulting object arrays share string objects, so
    duplicated names cost one pointer each.  Raw text is dropped afterwards."""
    n_jobs = n_workers(n_jobs)
    names = df["business_name"].astype(str)
    codes, uniq = pd.factorize(names)
    log(f"  normalising {len(uniq):,} unique names")
    _G["vals"] = np.asarray(uniq, dtype=object)
    res = _run(_name_worker, len(uniq), n_jobs)
    _assign(df, "n_", NAME_FIELDS, ("indic", "domain", "junk"), res, codes)
    del res, codes, uniq, names

    key = pd.MultiIndex.from_arrays([df["business_address"].astype(str),
                                     df["country"].astype(str)])
    codes, uniq = pd.factorize(key)
    log(f"  normalising {len(uniq):,} unique addresses")
    _G["vals"] = np.asarray(uniq.get_level_values(0), dtype=object)
    _G["ctry"] = np.asarray(uniq.get_level_values(1), dtype=object)
    res = _run(_addr_worker, len(uniq), n_jobs)
    _assign(df, "a_", ADDR_FIELDS, ("missing",), res, codes)
    _G.clear()
    if drop_raw:
        df.drop(columns=["business_name", "business_address"], inplace=True)
    free()
    return df


def load_normalized(data_dir, split, cache_dir=None, n_jobs=None, s1_ids=None, log=_log):
    """load_split + normalize_frame with an optional parquet/pickle cache so a
    re-run of the notebook skips the expensive normalisation."""
    cache = Path(cache_dir) / f"norm_{split}" if cache_dir else None
    if cache is not None and (cache / "tgt.pkl").exists():
        log(f"loading normalised {split} from cache {cache}")
        s1 = pd.read_pickle(cache / "s1.pkl")
        tgt = pd.read_pickle(cache / "tgt.pkl")
        gt = pd.read_pickle(cache / "gt.pkl") if (cache / "gt.pkl").exists() else None
    else:
        log(f"loading {split} from {data_dir}")
        s1, tgt, gt = load_split(data_dir, split)
        log(f"  S1={len(s1):,} targets={len(tgt):,}")
        log("normalising S1 ...")
        normalize_frame(s1, n_jobs, log)
        log("normalising targets ...")
        normalize_frame(tgt, n_jobs, log)
        if cache is not None:
            cache.mkdir(parents=True, exist_ok=True)
            s1.to_pickle(cache / "s1.pkl")
            tgt.to_pickle(cache / "tgt.pkl")
            if gt is not None:
                gt.to_pickle(cache / "gt.pkl")
    if s1_ids is not None:
        s1 = s1[s1["entity_id"].isin(s1_ids)].reset_index(drop=True)
    return s1, tgt, gt

Writing entity_matching/data.py


In [7]:
%%writefile entity_matching/blocking.py
"""
Candidate generation (blocking).

Country is a HARD block (cell 9: 100 % of true links are same-country).
Inside a country every record emits hashed keys of several types; an S1
record is paired with every target sharing one of its rarest keys and the
pair is scored by the IDF-weighted sum of shared keys.  We keep the top-K
pairs per S1 by total score plus the top-K2 by name-only and address-only
score, so a record whose name is unrecognisable (cell 17: names replaced by
"Kelopyrahalo") can still be found through its address, and vice versa.

EDA motivation: name token OR number gives 96.9 % recall and adding rare
address tokens 99.7 % (cells 21-23) - but with capped fan-out.
"""
from __future__ import annotations

from dataclasses import dataclass, field
import numpy as np
import pandas as pd

from .normalize import skeleton
from .utils import free, run_pool

_G: dict = {}

# key type -> (weight, max document frequency, max keys per record, group)
# group: 0 = name evidence, 1 = address evidence
KEY_TYPES = {
    "N": (1.0, 2000, 6, 0),   # skeleton name token
    "B": (1.6, 2000, 4, 0),   # skeleton name bigram
    "P": (1.6, 1000, 2, 0),   # first 8 chars of space-less core (domains, spacing)
    "F": (3.0, 2000, 1, 0),   # full skeleton name
    "A": (0.6, 3000, 6, 1),   # address word
    "U": (1.0, 3000, 6, 1),   # address number (canonical)
    "S": (2.2, 1000, 4, 1),   # number + following street word
    "C": (2.5, 1000, 6, 1),   # name token x address number ("arihant|417")
}
TYPE_CODE = {k: i for i, k in enumerate(KEY_TYPES)}


@dataclass
class BlockConfig:
    top_k: int = 40
    top_k_name: int = 10
    top_k_addr: int = 10
    pair_budget: int = 5_000_000  # max expanded postings per chunk (~0.6 GB temp)
    max_df_frac: float = 0.01
    n_jobs: int = 4
    key_types: dict = field(default_factory=lambda: dict(KEY_TYPES))


def _keys_for(skel, alts, nospace, alpha, nums, numstreet):
    out = []
    toks = [t for t in skel.split() if len(t) >= 2]
    for t in dict.fromkeys(toks):
        out.append(("N", t))
    for a, b in zip(toks, toks[1:]):
        out.append(("B", a + "_" + b))
    if alts:
        for alt in alts.split("||"):
            for t in alt.split():
                s = skeleton(t)
                if len(s) >= 2:
                    out.append(("N", s))
            ns = alt.replace(" ", "")
            if len(ns) >= 5:
                out.append(("P", ns[:8]))
    if len(nospace) >= 5:
        out.append(("P", nospace[:8]))
    if skel:
        out.append(("F", skel.replace(" ", "")))
    for t in dict.fromkeys(alpha.split()):
        if len(t) >= 3:
            out.append(("A", t))
    for t in dict.fromkeys(nums.split()):
        out.append(("U", t))
    for t in dict.fromkeys(numstreet.split()):
        out.append(("S", t))
    # composite keys stay rare even when the name word and the number are
    # both common (generic names such as "pediatric group", EDA cell 20)
    num_list = list(dict.fromkeys(nums.split()))[:4]
    for t in list(dict.fromkeys(toks))[:4]:
        for n in num_list:
            out.append(("C", t + "|" + n))
    return out


def _key_worker(b):
    lo, hi = b
    cols = [c[lo:hi] for c in _G["cols"]]
    idx, codes, strs = [], [], []
    for i, row in enumerate(zip(*cols)):
        for typ, val in _keys_for(*row):
            idx.append(lo + i)
            codes.append(TYPE_CODE[typ])
            strs.append(typ + ":" + val)
    h = pd.util.hash_array(np.array(strs, dtype=object)) if strs else np.zeros(0, np.uint64)
    return (np.asarray(idx, np.int32), np.asarray(codes, np.int8), h)


def build_keys(df: pd.DataFrame, n_jobs: int = 4, chunk: int = 50000):
    """Hashed blocking keys -> (record index int32, key type int8, hash uint64)."""
    cols_names = ["n_skel", "n_alts", "n_nospace", "a_alpha", "a_nums", "a_numstreet"]
    _G["cols"] = [df[c].to_numpy() for c in cols_names]  # zero-copy, fork-shared
    res = run_pool(_key_worker, [(s, min(s + chunk, len(df))) for s in range(0, len(df), chunk)], n_jobs)
    _G.clear()
    if not res:
        return np.zeros(0, np.int32), np.zeros(0, np.int8), np.zeros(0, np.uint64)
    out = (np.concatenate([r[0] for r in res]),
           np.concatenate([r[1] for r in res]),
           np.concatenate([r[2] for r in res]))
    del res
    return out


def _topk_mask(s, score, k):
    """Boolean mask of the top-k rows by score within each s (s sorted not required)."""
    order = np.lexsort((-score, s))
    s_sorted = s[order]
    first = np.r_[True, s_sorted[1:] != s_sorted[:-1]]
    grp_start = np.maximum.accumulate(np.where(first, np.arange(len(s_sorted)), 0))
    rank = np.arange(len(s_sorted)) - grp_start
    mask = np.zeros(len(s), bool)
    mask[order[rank < k]] = True
    return mask


def generate_candidates(s1: pd.DataFrame, tgt: pd.DataFrame, cfg: BlockConfig,
                        s1_rows: np.ndarray | None = None, log=print) -> pd.DataFrame:
    """Returns DataFrame[s_row, t_row, blk_score, blk_name, blk_addr, blk_nkeys]."""
    kt = cfg.key_types
    weights = np.array([v[0] for v in kt.values()])
    caps = np.array([v[1] for v in kt.values()])
    per_rec = np.array([v[2] for v in kt.values()])
    group = np.array([v[3] for v in kt.values()])

    if s1_rows is None:
        s1_rows = np.arange(len(s1))
    out = []
    s_ctry = s1["country"].astype(str).to_numpy()
    t_ctry = tgt["country"].astype(str).to_numpy()
    for country in sorted(pd.unique(s_ctry[s1_rows])):
        s_rows = s1_rows[s_ctry[s1_rows] == country]
        t_rows = np.flatnonzero(t_ctry == country)
        if len(s_rows) == 0 or len(t_rows) == 0:
            continue
        log(f"  [{country}] S1={len(s_rows):,} targets={len(t_rows):,}")
        t_idx, t_typ, t_h = build_keys(tgt.iloc[t_rows], cfg.n_jobs)
        # de-duplicate (record, key) then document frequencies
        pk = pd.DataFrame({"i": t_idx, "h": t_h, "c": t_typ}).drop_duplicates(["i", "h"])
        t_idx, t_h, t_typ = pk["i"].to_numpy(), pk["h"].to_numpy(), pk["c"].to_numpy()
        uniq, inv, cnt = np.unique(t_h, return_inverse=True, return_counts=True)
        # caps are absolute for the full data and shrink with small target sets
        eff_caps = np.maximum(20, np.minimum(caps, cfg.max_df_frac * len(t_rows))).astype(np.int64)
        keep = cnt[inv] <= eff_caps[t_typ]
        t_idx, t_h = t_idx[keep], t_h[keep]
        del pk, uniq, inv, cnt, keep
        order = np.argsort(t_h, kind="stable")
        post_h, post_t = t_h[order], t_idx[order].astype(np.int32)
        del t_h, t_idx, t_typ, order
        ph_uniq, ph_start, ph_len = np.unique(post_h, return_index=True, return_counts=True)
        del post_h
        free()
        n_t = len(t_rows)

        if len(ph_uniq) == 0:
            continue
        s_idx, s_typ, s_h = build_keys(s1.iloc[s_rows], cfg.n_jobs)
        sk = pd.DataFrame({"i": s_idx, "h": s_h, "c": s_typ}).drop_duplicates(["i", "h"])
        pos = np.clip(np.searchsorted(ph_uniq, sk["h"].to_numpy()), 0, len(ph_uniq) - 1)
        found = ph_uniq[pos] == sk["h"].to_numpy()
        sk = sk[found].assign(p=pos[found])
        sk["df"] = ph_len[sk["p"].to_numpy()]
        sk = sk.sort_values(["i", "c", "df"])
        sk = sk[sk.groupby(["i", "c"]).cumcount().to_numpy() < per_rec[sk["c"].to_numpy()]]
        sk["w"] = weights[sk["c"].to_numpy()] * np.log1p(n_t / sk["df"].to_numpy())
        sk["g"] = group[sk["c"].to_numpy()]

        # chunk S1 records so each chunk expands to <= pair_budget postings:
        # RAM stays bounded whatever the key frequencies are
        si_all = sk["i"].to_numpy()
        cum = np.cumsum(ph_len[sk["p"].to_numpy()])
        rec_start = np.flatnonzero(np.r_[True, si_all[1:] != si_all[:-1]])
        rec_cum = np.r_[0, cum][rec_start]
        cuts = rec_start[np.unique(np.searchsorted(
            rec_cum, np.arange(0, (cum[-1] if len(cum) else 0) + cfg.pair_budget, cfg.pair_budget)))
            .clip(0, len(rec_start) - 1)] if len(rec_start) else np.zeros(0, np.int64)
        bounds = np.unique(np.r_[cuts, len(sk)])
        bounds = np.r_[0, bounds[bounds > 0]] if len(bounds) else np.zeros(1, np.int64)
        for b0, b1 in zip(bounds[:-1], bounds[1:]):
            if b1 <= b0:
                continue
            ch = sk.iloc[b0:b1]
            starts = ph_start[ch["p"].to_numpy()]
            lens = ph_len[ch["p"].to_numpy()]
            total = int(lens.sum())
            if total == 0:
                continue
            rep = np.repeat(np.arange(len(ch)), lens)
            offs = np.arange(total) - np.repeat(np.cumsum(lens) - lens, lens)
            tt = post_t[np.repeat(starts, lens) + offs]
            ss = ch["i"].to_numpy()[rep]
            w = ch["w"].to_numpy()[rep]
            g = ch["g"].to_numpy()[rep]
            pair = ss.astype(np.int64) * (n_t + 1) + tt
            up, pinv = np.unique(pair, return_inverse=True)
            score = np.bincount(pinv, weights=w)
            sname = np.bincount(pinv, weights=w * (g == 0))
            saddr = np.bincount(pinv, weights=w * (g == 1))
            nkeys = np.bincount(pinv)
            ps = up // (n_t + 1)
            pt = up % (n_t + 1)
            m = (_topk_mask(ps, score, cfg.top_k)
                 | _topk_mask(ps, sname, cfg.top_k_name)
                 | _topk_mask(ps, saddr, cfg.top_k_addr))
            out.append(pd.DataFrame({
                "s_row": s_rows[ps[m]].astype(np.int32), "t_row": t_rows[pt[m]].astype(np.int32),
                "blk_score": score[m].astype(np.float32),
                "blk_name": sname[m].astype(np.float32),
                "blk_addr": saddr[m].astype(np.float32),
                "blk_nkeys": nkeys[m].astype(np.int16),
            }))
        del post_t, ph_uniq, ph_start, ph_len, sk
        free()
    if not out:
        return pd.DataFrame(columns=["s_row", "t_row", "blk_score", "blk_name", "blk_addr", "blk_nkeys"])
    return pd.concat(out, ignore_index=True)

Writing entity_matching/blocking.py


In [8]:
%%writefile entity_matching/features.py
"""
Pairwise comparison features for (S1, target) candidate pairs.

Groups
------
name      : fuzzy ratios on several normalised views (core / OCR-folded /
            phonetic skeleton / space-less), IDF-weighted token overlap,
            alias (DBA) best match, domain containment, acronym, legal form.
address   : fuzzy ratios, IDF-weighted word overlap, number agreement
            (exact / compound / parts / house number / number+street),
            state agreement, missing flags.
rules     : hard-coded boolean patterns (exact core + shared number, ...)
context   : rank / gap of the pair among the S1's candidates and among the
            S1 records competing for the same target (one-owner, cell 8).
"""
from __future__ import annotations

import math
from collections import Counter
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

from .utils import run_pool

PAIR_FEATURES = [
    # name
    "n_ratio_full", "n_ratio_core", "n_tsort_core", "n_tset_core", "n_partial_core",
    "n_wratio_core", "n_jw_nospace", "n_ratio_ocr", "n_tset_ocr", "n_ratio_skel",
    "n_tset_skel", "n_idf_jacc", "n_idf_contain", "n_idf_shared_max",
    "n_idf_miss_s", "n_idf_miss_t", "n_alt_best", "n_nospace_contain",
    "n_nospace_partial", "n_acr", "n_legal_both", "n_legal_eq", "n_legal_conflict",
    "n_ntok_s", "n_ntok_t", "n_ntok_diff", "n_exact_core", "n_exact_skel",
    "n_core_df_t", "s_indic", "t_indic", "t_domain", "t_junk",
    # address
    "a_missing_t", "a_ratio", "a_tset", "a_tsort", "a_partial", "a_idf_jacc",
    "a_idf_contain", "a_ratio_alpha_nospace", "num_s", "num_t", "num_shared",
    "num_jacc", "num_t_only", "num_s_only", "num_compound_shared", "numparts_shared",
    "numparts_jacc", "first_num_eq", "numstreet_shared", "state_eq", "postal_eq",
    # rules
    "r_exact_core_num", "r_skel_num", "r_domain_num", "r_alias_num",
    "r_name_only_strong", "r_addr_only_strong",
    # misc
    "is_s3",
]
BLOCK_FEATURES = ["blk_score", "blk_name", "blk_addr", "blk_nkeys"]
CONTEXT_FEATURES = [
    "ctx_ncand_s", "ctx_rank_s", "ctx_gap_s", "ctx_ncand_t", "ctx_rank_t", "ctx_gap_t",
    "ctx_blk_rank_s", "ctx_same_core_in_s", "ctx_same_num_in_s",
]
ALL_FEATURES = PAIR_FEATURES + BLOCK_FEATURES + CONTEXT_FEATURES

_LEGAL_EQUIV = [{"llc", "pllc", "lc"}, {"inc", "corp"}, {"ltd", "pvt", "public", "plc"},
                {"llp", "lp"}, {"pc", "pa"}, {"sarl", "eurl"}, {"sas", "sasu", "sa"}]

# globals shared with forked workers
_G: dict = {}


def build_idf(tgt: pd.DataFrame) -> dict:
    """Per-country IDF tables from TARGET records (never from S1/test labels)."""
    idf = {}
    for country, grp in tgt.groupby(tgt["country"].astype(str), observed=True):
        n = len(grp)
        cn, ca = Counter(), Counter()
        for s in grp["n_skel"]:
            cn.update(set(s.split()))
        for s in grp["a_alpha"]:
            ca.update(set(s.split()))
        core_df = Counter(grp["n_skel"])
        idf[country] = {
            "n": {k: math.log1p(n / v) for k, v in cn.items()},
            "a": {k: math.log1p(n / v) for k, v in ca.items()},
            "core_df": dict(core_df),
            "default": math.log1p(n),
        }
    return idf


def _idf_overlap(a_toks, b_toks, table, default):
    if not a_toks or not b_toks:
        return 0.0, 0.0, 0.0, 1.0, 1.0
    sa, sb = set(a_toks), set(b_toks)
    w = lambda t: table.get(t, default)  # noqa: E731
    inter = sa & sb
    wi = sum(w(t) for t in inter)
    wa = sum(w(t) for t in sa)
    wb = sum(w(t) for t in sb)
    wu = wa + wb - wi
    return (wi / wu if wu else 0.0,
            wi / min(wa, wb) if min(wa, wb) else 0.0,
            max((w(t) for t in inter), default=0.0),
            (wa - wi) / wa if wa else 1.0,
            (wb - wi) / wb if wb else 1.0)


def _legal_rel(a, b):
    sa, sb = set(a.split()), set(b.split())
    both = int(bool(sa) and bool(sb))
    eq = int(both and sa == sb)
    conflict = 0
    if both and not (sa & sb):
        conflict = 1
        for grp in _LEGAL_EQUIV:
            if sa & grp and sb & grp:
                conflict = 0
    return both, eq, conflict


def _pair(i, j):
    S, T = _G["S"], _G["T"]
    country = str(S["country"][i])
    tab = _G["idf"].get(country) or {"n": {}, "a": {}, "core_df": {}, "default": 10.0}
    d = tab["default"]

    sc, tc = S["n_core"][i], T["n_core"][j]
    sf, tf = S["n_full"][i], T["n_full"][j]
    so, to = S["n_ocr"][i], T["n_ocr"][j]
    ss, ts = S["n_skel"][i], T["n_skel"][j]
    sn, tn = S["n_nospace"][i], T["n_nospace"][j]
    stoks, ttoks = ss.split(), ts.split()

    jacc, contain, smax, miss_s, miss_t = _idf_overlap(stoks, ttoks, tab["n"], d)

    alt_best = 0.0
    for alt in (T["n_alts"][j].split("||") if T["n_alts"][j] else []):
        alt_best = max(alt_best, fuzz.token_set_ratio(sc, alt))
    for alt in (S["n_alts"][i].split("||") if S["n_alts"][i] else []):
        alt_best = max(alt_best, fuzz.token_set_ratio(alt, tc))

    short, long_ = (sn, tn) if len(sn) <= len(tn) else (tn, sn)
    nospace_contain = int(len(short) >= 4 and short in long_)
    acr = int(bool(S["n_acr"][i] and S["n_acr"][i] == tn) or bool(T["n_acr"][j] and T["n_acr"][j] == sn))
    lboth, leq, lconf = _legal_rel(S["n_legal"][i], T["n_legal"][j])

    # address
    sa_txt, ta_txt = S["a_atext"][i], T["a_atext"][j]
    salpha, talpha = S["a_alpha"][i].split(), T["a_alpha"][j].split()
    ajacc, acontain, _, _, _ = _idf_overlap(salpha, talpha, tab["a"], d)
    snum, tnum = set(S["a_nums"][i].split()), set(T["a_nums"][j].split())
    shared = snum & tnum
    union = snum | tnum
    comp_shared = sum(1 for x in shared if ("/" in x or "-" in x or not x.isdigit()))
    sp, tp = set(S["a_numparts"][i].split()), set(T["a_numparts"][j].split())
    sp_sh = len(sp & tp)
    sp_un = len(sp | tp)
    sns, tns = set(S["a_numstreet"][i].split()), set(T["a_numstreet"][j].split())
    sst, tst = S["a_state"][i], T["a_state"][j]
    state_eq = (1 if set(sst.split()) & set(tst.split()) else -1) if sst and tst else 0
    spo, tpo = S["a_postal"][i], T["a_postal"][j]
    postal_eq = (1 if set(spo.split()) & set(tpo.split()) else -1) if spo and tpo else 0
    a_missing = int(T["a_missing"][j])

    n_tset_skel = fuzz.token_set_ratio(ss, ts)
    a_tset = fuzz.token_set_ratio(sa_txt, ta_txt) if not a_missing else -1.0
    exact_core = int(sc == tc and sc != "")
    exact_skel = int(ss == ts and ss != "")
    n_shared = len(shared)

    return (
        fuzz.ratio(sf, tf), fuzz.ratio(sc, tc), fuzz.token_sort_ratio(sc, tc),
        fuzz.token_set_ratio(sc, tc), fuzz.partial_ratio(sc, tc), fuzz.WRatio(sc, tc),
        JaroWinkler.normalized_similarity(sn, tn) * 100, fuzz.ratio(so, to),
        fuzz.token_set_ratio(so, to), fuzz.ratio(ss, ts), n_tset_skel,
        jacc, contain, smax, miss_s, miss_t, alt_best, nospace_contain,
        fuzz.partial_ratio(sn, tn), acr, lboth, leq, lconf,
        len(stoks), len(ttoks), abs(len(stoks) - len(ttoks)), exact_core, exact_skel,
        math.log1p(tab["core_df"].get(ts, 0)), S["n_indic"][i], T["n_indic"][j],
        T["n_domain"][j], T["n_junk"][j],
        a_missing,
        fuzz.ratio(sa_txt, ta_txt) if not a_missing else -1.0,
        a_tset,
        fuzz.token_sort_ratio(sa_txt, ta_txt) if not a_missing else -1.0,
        fuzz.partial_ratio(sa_txt, ta_txt) if not a_missing else -1.0,
        ajacc, acontain,
        fuzz.ratio("".join(sorted(salpha)), "".join(sorted(talpha))) if not a_missing else -1.0,
        len(snum), len(tnum), n_shared, n_shared / len(union) if union else 0.0,
        len(tnum - snum), len(snum - tnum), comp_shared, sp_sh, sp_sh / sp_un if sp_un else 0.0,
        int(bool(S["a_first_num"][i]) and S["a_first_num"][i] == T["a_first_num"][j]),
        len(sns & tns), state_eq, postal_eq,
        # rules
        int(exact_core and n_shared > 0), int(exact_skel and n_shared > 0),
        int(T["n_domain"][j] and nospace_contain and n_shared > 0),
        int(alt_best >= 95 and n_shared > 0),
        int(n_tset_skel >= 95 and contain >= 0.9),
        int((a_tset >= 90) and n_shared >= 2),
        int(str(T["source"][j]) == "S3"),
    )


def _worker(bounds):
    lo, hi = bounds
    si, ti = _G["si"], _G["ti"]
    out = np.empty((hi - lo, len(PAIR_FEATURES)), np.float32)
    for r in range(lo, hi):
        out[r - lo] = _pair(si[r], ti[r])
    return lo, out


_S_COLS = ["country", "n_core", "n_full", "n_ocr", "n_skel", "n_nospace", "n_alts",
           "n_acr", "n_legal", "n_indic", "a_atext", "a_alpha", "a_nums", "a_numparts",
           "a_numstreet", "a_state", "a_postal", "a_first_num", "a_missing"]
_T_COLS = _S_COLS + ["n_domain", "n_junk", "source"]


def compute_pair_features(cands: pd.DataFrame, s1: pd.DataFrame, tgt: pd.DataFrame,
                          idf: dict, n_jobs: int = 4, chunk: int = 20000) -> pd.DataFrame:
    # zero-copy numpy views, shared with forked workers (nothing is pickled)
    _G["S"] = {c: s1[c].to_numpy() for c in _S_COLS}
    _G["T"] = {c: tgt[c].to_numpy() for c in _T_COLS}
    _G["idf"] = idf
    _G["si"] = cands["s_row"].to_numpy()
    _G["ti"] = cands["t_row"].to_numpy()
    n = len(cands)
    X = np.empty((n, len(PAIR_FEATURES)), np.float32)
    for lo, arr in run_pool(_worker, [(lo, min(lo + chunk, n)) for lo in range(0, n, chunk)], n_jobs):
        X[lo:lo + len(arr)] = arr
    _G.clear()
    feats = pd.DataFrame(X, columns=PAIR_FEATURES, index=cands.index)
    return pd.concat([cands, feats], axis=1)


def add_target_context(cands: pd.DataFrame) -> pd.DataFrame:
    """Competition for the same target across ALL S1 records (one-owner
    constraint).  Uses blocking scores so it can be computed globally before
    the batched feature pass."""
    g = cands.groupby("t_row")["blk_score"]
    cands["ctx_ncand_t"] = g.transform("size").astype(np.float32)
    cands["ctx_rank_t"] = g.rank(ascending=False, method="min").astype(np.float32)
    cands["ctx_gap_t"] = (g.transform("max") - cands["blk_score"]).astype(np.float32)
    return cands


def add_s1_context(df: pd.DataFrame, tgt: pd.DataFrame) -> pd.DataFrame:
    """Ranking features within each S1's candidate list (needs all candidates
    of an S1 in the same batch)."""
    comb = (df["n_tset_skel"].to_numpy() + df["n_idf_contain"].to_numpy() * 100
            + np.clip(df["a_tset"].to_numpy(), 0, None) + 20 * df["num_jacc"].to_numpy())
    df["_comb"] = comb
    gs = df.groupby("s_row")["_comb"]
    df["ctx_ncand_s"] = gs.transform("size").astype(np.float32)
    df["ctx_rank_s"] = gs.rank(ascending=False, method="min").astype(np.float32)
    df["ctx_gap_s"] = (gs.transform("max") - comb).astype(np.float32)
    df["ctx_blk_rank_s"] = df.groupby("s_row")["blk_score"].rank(
        ascending=False, method="min").astype(np.float32)
    # siblings: an S1 usually has several S2/S3 duplicates of itself
    df["_core"] = tgt["n_skel"].to_numpy()[df["t_row"].to_numpy()]
    df["_fnum"] = tgt["a_first_num"].to_numpy()[df["t_row"].to_numpy()]
    df["ctx_same_core_in_s"] = df.groupby(["s_row", "_core"])["_comb"].transform("size").astype(np.float32)
    df["ctx_same_num_in_s"] = df.groupby(["s_row", "_fnum"])["_comb"].transform("size").astype(np.float32)
    df.loc[df["_fnum"] == "", "ctx_same_num_in_s"] = 0
    return df.drop(columns=["_comb", "_core", "_fnum"])

Writing entity_matching/features.py


In [9]:
%%writefile entity_matching/metrics.py
"""F-beta evaluation (default beta = 0.5: precision weighted 2x recall)."""
from __future__ import annotations

import numpy as np
import pandas as pd


def fbeta(p: float, r: float, beta: float = 0.5) -> float:
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r) if (p + r) > 0 else 0.0


def evaluate(pred: pd.DataFrame, truth: pd.DataFrame, s1_ids, beta: float = 0.5) -> dict:
    """pred / truth: DataFrames[source1_entity_id, matched_id].
    Only S1 ids in `s1_ids` are scored.  Returns micro (pair-level) and macro
    (per-S1 averaged, empty-vs-empty = 1) scores."""
    s1_ids = pd.Index(pd.unique(np.asarray(s1_ids)))
    pred = pred[pred["source1_entity_id"].isin(s1_ids)]
    truth = truth[truth["source1_entity_id"].isin(s1_ids)]
    pk = set(zip(pred["source1_entity_id"], pred["matched_id"]))
    tk = set(zip(truth["source1_entity_id"], truth["matched_id"]))
    tp = len(pk & tk)
    p = tp / len(pk) if pk else 1.0
    r = tp / len(tk) if tk else 1.0

    # per-S1
    tp_s = pd.Series([a for a, _ in pk & tk], dtype=object).value_counts()
    np_s = pred["source1_entity_id"].value_counts()
    nt_s = truth["source1_entity_id"].value_counts()
    tp_v = tp_s.reindex(s1_ids).fillna(0).to_numpy()
    np_v = np_s.reindex(s1_ids).fillna(0).to_numpy()
    nt_v = nt_s.reindex(s1_ids).fillna(0).to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        ps = np.where(np_v > 0, tp_v / np_v, np.where(nt_v > 0, 0.0, 1.0))
        rs = np.where(nt_v > 0, tp_v / nt_v, np.where(np_v > 0, 0.0, 1.0))
        b2 = beta * beta
        fs = np.where((ps + rs) > 0, (1 + b2) * ps * rs / (b2 * ps + rs), 0.0)
    both_empty = (np_v == 0) & (nt_v == 0)
    fs = np.where(both_empty, 1.0, fs)
    return {
        "micro_precision": p, "micro_recall": r, "micro_fbeta": fbeta(p, r, beta),
        "macro_precision": float(ps.mean()), "macro_recall": float(rs.mean()),
        "macro_fbeta": float(fs.mean()), "tp": tp, "n_pred": len(pk), "n_true": len(tk),
    }

Writing entity_matching/metrics.py


In [10]:
%%writefile entity_matching/synth.py
"""
Synthetic data in the exact challenge layout, reproducing the noise found
in the EDA notebook.  Used for smoke-testing the pipeline end-to-end (the
real Kaggle data is not bundled).  Scores on synthetic data are NOT an
estimate of leaderboard scores.
"""
from __future__ import annotations

import random
from pathlib import Path

import pandas as pd

US_W1 = ["Summit", "Harbor", "Meridian", "Pioneer", "Cascade", "Sterling", "Keystone", "Apex",
         "Coastal", "Highland", "Silver", "Golden", "Bright", "Heritage", "Metro", "Blue",
         "Liberty", "Frontier", "Evergreen", "Oakwood", "Riverside", "Granite", "Beacon", "Vista"]
US_W2 = ["Pediatric", "Dental", "Eye", "Chiropractic", "Family", "Urology", "Vision",
         "Orthopedic", "Seafood", "Auto Body", "Printing", "Realty", "Capital", "Logistics",
         "Consulting", "Cargo", "Marketing", "Software", "Roofing", "Insurance"]
US_W3 = ["Group", "Associates", "Partners", "Center", "Care", "Clinic", "Services", "Holdings",
         "Solutions", "Works"]
US_LEGAL = ["LLC", "Inc", "Corp", "Co", "LP", "PC", "PLLC", "Ltd", ""]
SURNAMES = ["Smith", "Jones", "Garcia", "Kern", "Delavega", "Hawkins", "Brunson", "Murphy",
            "Townsend", "Graham", "Figueroa", "Prater", "Younker", "Wellman", "Holloway"]
US_STREETS = ["Westchester", "Ellis", "Montebello", "Cameron", "Elm", "Stardust", "Pierpont",
              "Mulberry", "Forest", "Moore", "Greyrock", "Meadow", "Allison", "Bell", "Oak",
              "Maple", "Cedar", "Lincoln", "Washington", "Park"]
US_TYPES = [("Street", "St"), ("Road", "Rd"), ("Drive", "Dr"), ("Avenue", "Ave"),
            ("Lane", "Ln"), ("Court", "Ct"), ("Trail", "Trl"), ("Boulevard", "Blvd")]
US_CITIES = [("High Point", "NC", "North Carolina"), ("Tahlequah", "OK", "Oklahoma"),
             ("Phoenix", "AZ", "Arizona"), ("Dundalk", "MD", "Maryland"),
             ("Cleveland", "OH", "Ohio"), ("Houston", "TX", "Texas"),
             ("Brooklyn", "NY", "New York"), ("Springfield", "IL", "Illinois"),
             ("Raleigh", "NC", "North Carolina"), ("Eugene", "OR", "Oregon")]

IN_W1 = ["Shakti", "Lakshmi", "Supreme", "Sky", "Green", "Arihant", "Jain", "Global", "Royal",
         "Modern", "Sun", "Krishna", "Balaji", "Great", "Prime", "Apex", "Shivam", "Om"]
IN_W2 = ["Technology", "Builders", "Consultants", "Foods", "Infra", "Marketing", "Ventures",
         "Enterprises", "Exports", "Solutions", "Estate", "Agro", "Finance"]
IN_LEGAL = ["Private Limited", "Pvt Ltd", "Limited", "LLP", "Pvt. Ltd.", ""]
DEVANAGARI = {
    "Shakti": "शक्ति", "Lakshmi": "लक्ष्मी", "Supreme": "सुप्रीम", "Sky": "स्काई",
    "Green": "ग्रीन", "Arihant": "अरिहंत", "Jain": "जैन", "Global": "ग्लोबल", "Royal": "रॉयल",
    "Modern": "मॉडर्न", "Sun": "सन", "Krishna": "कृष्णा", "Balaji": "बालाजी", "Great": "ग्रेट",
    "Prime": "प्राइम", "Apex": "एपेक्स", "Shivam": "शिवम", "Om": "ओम",
    "Technology": "टेक्नोलॉजी", "Builders": "बिल्डर्स", "Consultants": "कंसल्टेंट्स",
    "Foods": "फूड्स", "Infra": "इंफ्रा", "Marketing": "मार्केटिंग", "Ventures": "वेंचर्स",
    "Enterprises": "एंटरप्राइजेज", "Exports": "एक्सपोर्ट्स", "Solutions": "सॉल्यूशंस",
    "Estate": "एस्टेट", "Agro": "एग्रो", "Finance": "फाइनेंस",
    "Private": "प्राइवेट", "Limited": "लिमिटेड", "Pvt": "प्रा.", "Ltd": "लि.", "LLP": "एलएलपी",
}
IN_AREAS = ["Gulmohar Colony", "Karve Nagar", "Anand Nagar", "Jayanagar", "Kamala Nagar",
            "Andheri East", "Sector 42", "Rajaji Nagar", "Salt Lake", "Banjara Hills"]
IN_CITIES = [("Mumbai", "MH", "Maharashtra", "महाराष्ट्र"), ("Pune", "MH", "Maharashtra", "महाराष्ट्र"),
             ("Bangalore", "KA", "Karnataka", "ಕರ್ನಾಟಕ"), ("Chennai", "TN", "Tamil Nadu", "தமிழ்நாடு"),
             ("Hyderabad", "TG", "Telangana", "తెలంగాణ"), ("Kolkata", "WB", "West Bengal", "পশ্চিমবঙ্গ"),
             ("Ahmedabad", "GJ", "Gujarat", "ગુજરાત"), ("New Delhi", "DL", "Delhi", "दिल्ली")]
IN_LABELS = ["No.", "H.No", "Plot No", "Door No", "Flat No", "Shop No", "Office No"]

FR_W1 = ["Amicale", "Club", "Ecole", "Maison", "Comite", "Union", "Groupe", "Centre",
         "Pharmacie", "Institut", "Association", "Foyer"]
FR_W2 = ["Sportive", "Culturelle", "des Parents", "Bouliste", "du Lac", "Saint-Michel",
         "Laique", "Rotary", "des Amis", "Primaire"]
FR_LEGAL = ["SARL", "SAS", "SASU", "EURL", "SA", "SCI", ""]
FR_STREETS = ["Pierre Dignac", "Parmentier", "Lachassaigne", "de Cassel", "du Chaufour",
              "Jules Lefebvre", "Roger Salengro", "du President Wilson", "de la Renaudiere"]
FR_TYPES = [("Rue", "R"), ("Avenue", "Av"), ("Boulevard", "BD"), ("Allee", "All"),
            ("Impasse", "Imp"), ("Chemin", "Ch")]
FR_CITIES = [("Bordeaux", "Nouvelle-Aquitaine"), ("Nantes", "Pays de la Loire"),
             ("Lille", "Hauts-de-France"), ("Dunkerque", "Hauts-de-France"),
             ("Calais", "Hauts-de-France"), ("Pessac", "Nouvelle-Aquitaine")]

MATCH_COUNT_DIST = [(0, 5.6), (1, 5.4), (2, 17.0), (3, 24.1), (4, 21.9), (5, 14.6),
                    (6, 7.5), (7, 2.9), (8, 0.85), (9, 0.2), (10, 0.03)]
ACCENTS = {"a": "á", "e": "é", "i": "í", "o": "ó", "u": "ú", "c": "ç"}
OCR = {"o": "0", "l": "1", "s": "5", "i": "l", "e": "3"}


class Gen:
    def __init__(self, seed):
        self.r = random.Random(seed)
        self.used_ids = set()

    def uid(self, prefix):
        while True:
            i = self.r.randint(10_000, 999_999_999)
            if i not in self.used_ids:
                self.used_ids.add(i)
                return f"{prefix}-{i}"

    # ---------------- clean entities ----------------
    def entity(self, country):
        r = self.r
        if country == "US":
            if r.random() < 0.35:  # generic -> hard negatives (cell 20)
                words = [r.choice(US_W2[:8]), r.choice(["Group", "Associates", "Center", "Care"])]
            elif r.random() < 0.2:
                a, b = r.sample(SURNAMES, 2)
                words = [f"{a},", b, "&", r.choice(SURNAMES)]
            else:
                words = [r.choice(US_W1), r.choice(US_W2)] + ([r.choice(US_W3)] if r.random() < .6 else [])
            legal = r.choice(US_LEGAL)
            city, code, state = r.choice(US_CITIES)
            num = str(r.randint(1, 29999))
            st, _ = r.choice(US_TYPES)
            addr = {"num": num, "street": r.choice(US_STREETS), "type": st, "city": city,
                    "code": code, "state": state,
                    "unit": (f"Unit {r.randint(1, 900)}" if r.random() < 0.2 else "")}
        elif country == "India":
            words = [r.choice(IN_W1), r.choice(IN_W2)]
            if r.random() < 0.3:
                words.insert(1, r.choice(IN_W1))
            legal = r.choice(IN_LEGAL)
            city, code, state, native = r.choice(IN_CITIES)
            n = str(r.randint(1, 999))
            num = n + (f"/{r.randint(1, 99)}" if r.random() < 0.5 else "")
            addr = {"label": r.choice(IN_LABELS), "num": num, "area": r.choice(IN_AREAS),
                    "city": city, "code": code, "state": state, "native": native,
                    "floor": (r.choice(["1st Floor", "2nd Floor", "Ground Floor"]) if r.random() < .3 else "")}
        else:
            words = [r.choice(FR_W1), r.choice(FR_W2)]
            legal = r.choice(FR_LEGAL)
            city, region = r.choice(FR_CITIES)
            st, _ = r.choice(FR_TYPES)
            addr = {"num": str(r.randint(1, 400)), "bis": r.random() < 0.1, "type": st,
                    "street": r.choice(FR_STREETS), "city": city, "region": region}
        return {"country": country, "words": words, "legal": legal, "addr": addr}

    def render_name(self, e):
        return " ".join(e["words"] + ([e["legal"]] if e["legal"] else [])).replace(" ,", ",")

    def render_addr(self, e, noisy=False):
        r, a, c = self.r, e["addr"], e["country"]
        if c == "US":
            typ = a["type"]
            if noisy and r.random() < 0.5:
                typ = dict(US_TYPES)[typ]
            num = a["num"]
            if noisy and r.random() < 0.1:
                num = "0" + num
            if noisy and r.random() < 0.05:
                num = "#" + num
            state = a["code"] if (not noisy or r.random() < 0.5) else a["state"]
            comps = [f"{num} {a['street']} {typ}"] + ([a["unit"]] if a["unit"] else []) + [a["city"], state]
        elif c == "India":
            lab = a["label"] if not noisy else r.choice(IN_LABELS + [a["label"]] * 2)
            state = a["state"]
            if noisy:
                state = r.choice([a["state"], a["code"], a["native"]])
            comps = [f"{lab} {a['num']}"] + ([a["floor"]] if a["floor"] else []) + [a["area"], a["city"], state]
        else:
            typ = a["type"]
            if noisy and r.random() < 0.4:
                typ = dict(FR_TYPES)[typ]
            num = a["num"] + (" bis" if a["bis"] else "")
            comps = [f"{num} {typ} {a['street']}", a["city"], a["region"]]
        if noisy:
            if r.random() < 0.35:
                r.shuffle(comps)
            if r.random() < 0.08:
                comps.insert(r.randint(0, len(comps)), r.choice(["NULL", "null", "<NULL>"]))
            if r.random() < 0.1 and len(comps) > 2:
                comps.pop(r.randrange(len(comps)))
            s = ", ".join(comps)
            if r.random() < 0.3:
                s = s.upper()
            if r.random() < 0.15:
                s = self.typo(s)
            if r.random() < 0.035:
                s = ""
            return s
        return ", ".join(comps)

    # ---------------- noise ops ----------------
    def typo(self, s):
        r = self.r
        if len(s) < 4:
            return s
        i = r.randrange(1, len(s) - 1)
        op = r.random()
        if op < 0.3:
            return s[:i] + s[i + 1:]
        if op < 0.6:
            return s[:i] + r.choice("abcdefghilmnorstu") + s[i:]
        if op < 0.8:
            return s[:i] + s[i + 1] + s[i] + s[i + 2:]
        return s[:i] + r.choice("aeiou") + s[i + 1:]

    def noisy_name(self, e):
        r = self.r
        words = list(e["words"])
        legal = e["legal"]
        c = e["country"]
        if r.random() < 0.02:  # name replaced entirely (cell 17: "Kelopyrahalo")
            return r.choice(["Kelopyrahalo", "Noviveo", "Ectoecto", "Viovera", "Jaxzeta"])
        if c == "India" and r.random() < 0.2 and all(w in DEVANAGARI for w in words):
            parts = [DEVANAGARI[w] for w in words]
            for lw in legal.replace(".", "").split():
                parts.append(DEVANAGARI.get(lw, lw))
            if r.random() < 0.3:  # mixed script
                parts[0] = words[0]
            return " ".join(parts)
        ops = r.sample(range(14), k=r.choice([1, 1, 2, 2, 3]))
        for op in ops:
            if op == 0 and legal:
                legal = r.choice({"US": US_LEGAL, "India": IN_LEGAL, "France": FR_LEGAL}[c])
            elif op == 1 and legal:
                legal = r.choice([f"({legal})", f"[{legal}]", f"([{legal}])", legal.upper()])
            elif op == 2 and legal:
                words = [legal] + words
                legal = ""
            elif op == 3:
                i = r.randrange(len(words))
                words[i] = self.typo(words[i])
            elif op == 4:
                i = r.randrange(len(words))
                words[i] = "".join(OCR.get(ch, ch) if r.random() < 0.3 else ch for ch in words[i])
            elif op == 5:
                i = r.randrange(len(words))
                words[i] = "".join(ACCENTS.get(ch, ch) if r.random() < 0.2 else ch for ch in words[i])
            elif op == 6:
                i = r.randrange(len(words))
                words.insert(i, words[i])
            elif op == 7 and len(words) > 1:
                r.shuffle(words)
            elif op == 8:
                name = "".join(w.strip(",&") for w in words).lower()
                return name + r.choice([".com", ".Com", ".c0m", ".net"])
            elif op == 9:
                alias = r.choice(["Vantagexylo D.B.A.", "Korzephdova formerly:", "T/A"])
                return f"{alias} {' '.join(words)} {legal}".strip()
            elif op == 10:
                words.append(r.choice(["Partners", "Center", "Group", "Co", "Enterprises"]))
            elif op == 11:
                return f"{' '.join(words)} {legal} (ID: {r.randint(10000, 99999)})".strip()
            elif op == 12:
                words = [r.choice(["***", "--", ">>", "M/s"])] + words
            elif op == 13:
                legal = ""
        s = " ".join(words + ([legal] if legal else []))
        rr = r.random()
        if rr < 0.2:
            s = s.upper()
        elif rr < 0.3:
            s = s.lower()
        elif rr < 0.35:
            s = s.replace(" ", "  ", 1)
        return s

    # ---------------- dataset ----------------
    def build(self, n_s1, countries):
        s1, s2, s3, gt = [], [], [], []
        counts, weights = zip(*MATCH_COUNT_DIST)
        for _ in range(n_s1):
            c = self.r.choices(countries[0], weights=countries[1])[0]
            e = self.entity(c)
            sid = self.uid("S1")
            s1.append((sid, self.render_name(e), self.render_addr(e), c))
            k = self.r.choices(counts, weights=weights)[0]
            ids = []
            for _ in range(k):
                src = self.r.choice(["S2", "S3"])
                tid = self.uid(src)
                row = (tid, self.noisy_name(e), self.render_addr(e, noisy=True), c)
                (s2 if src == "S2" else s3).append(row)
                ids.append(tid)
            gt.append((sid, ",".join(ids)))
        # distractor targets: entities absent from S1 (~30 % of targets)
        n_dis = int(0.3 * (len(s2) + len(s3)))
        for _ in range(n_dis):
            c = self.r.choices(countries[0], weights=countries[1])[0]
            e = self.entity(c)
            src = self.r.choice(["S2", "S3"])
            row = (self.uid(src), self.noisy_name(e), self.render_addr(e, noisy=True), c)
            (s2 if src == "S2" else s3).append(row)
        cols = ["entity_id", "business_name", "business_address", "country"]
        self.r.shuffle(s2)
        self.r.shuffle(s3)
        return (pd.DataFrame(s1, columns=cols), pd.DataFrame(s2, columns=cols),
                pd.DataFrame(s3, columns=cols),
                pd.DataFrame(gt, columns=["source1_entity_id", "matched_entity_ids"]))


def write_synthetic(out_dir, n_train=6000, n_test=3000, seed=0):
    out = Path(out_dir)
    g = Gen(seed)
    for split, n, countries in [("train", n_train, (["US", "India"], [60, 40])),
                                ("test", n_test, (["US", "India", "France"], [38, 47, 15]))]:
        d = out / split
        d.mkdir(parents=True, exist_ok=True)
        s1, s2, s3, gt = g.build(n, countries)
        for name, df in [("source1", s1), ("source2", s2), ("source3", s3), ("ground_truth", gt)]:
            df.to_csv(d / f"{split}_{name}.tsv", sep="\t", index=False)
        print(f"{split}: S1={len(s1):,} S2={len(s2):,} S3={len(s3):,} links={gt.matched_entity_ids.str.count('S').sum():,}")

Writing entity_matching/synth.py


In [11]:
%%writefile entity_matching/run.py
"""
End-to-end pipeline (Kaggle-ready).

    python -m entity_matching.run kaggle                 # train + predict test, auto paths
    python -m entity_matching.run train   [--data-dir D] [--out-dir O]
    python -m entity_matching.run predict [--data-dir D] [--out-dir O]
    python -m entity_matching.run synth   --out-dir SYNTH_DATA

`--data-dir` is optional: the dataset is auto-detected under /kaggle/input
(any slug / nesting) or local folders.  Outputs go to /kaggle/working/em_model
on Kaggle (./em_output elsewhere); the submission is also copied to
/kaggle/working/submission.tsv.

`train` produces out-of-fold (GroupKFold by S1) predictions, tunes the
decision rule for F-beta (beta=0.5 by default), reports honest validation
scores on ALL held-out S1 records (singletons and doubtful pairs included,
true links missed by blocking counted as false negatives), and saves the
final model.
"""
from __future__ import annotations

import argparse
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from .blocking import BlockConfig, generate_candidates
from .data import gt_pairs, load_normalized
from .features import (ALL_FEATURES, add_s1_context, add_target_context,
                       build_idf, compute_pair_features)
from .metrics import evaluate, fbeta
from .utils import (auto_data_dir, default_out_dir, free, is_kaggle, log,
                    n_workers)

LGB_PARAMS = dict(
    objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=40,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    max_bin=255, verbose=-1, seed=42, force_col_wise=True,
)
DOUBT_COLS = ["n_tset_skel", "n_idf_contain", "a_tset", "num_shared", "n_alt_best",
              "n_exact_skel", "a_missing_t", "num_jacc", "num_s"]


# ---------------------------------------------------------------------------
# Shared steps
# ---------------------------------------------------------------------------
def build_pairs(s1, tgt, cfg, n_jobs, keep_s_rows=None, predict_fn=None,
                batch_s1=100_000, extra_fn=None):
    """Blocking over ALL S1 records (so target-competition features see every
    competitor, as at test time), then features in S1 batches.

    keep_s_rows : only these S1 rows get features (training sample)
    predict_fn  : part -> probabilities; then only (s_row, t_row, p) is kept
    extra_fn    : part -> dict of extra arrays to keep (used for training)"""
    log("blocking ...")
    cands = generate_candidates(s1, tgt, cfg, log=log)
    log(f"  candidates: {len(cands):,} ({len(cands) / max(len(s1), 1):.1f} per S1)")
    cands = add_target_context(cands)
    if keep_s_rows is not None:
        cands = cands[np.isin(cands["s_row"].to_numpy(), keep_s_rows)]
    log("building IDF tables from targets ...")
    idf = build_idf(tgt)
    log("pair features ...")
    cands = cands.sort_values("s_row", kind="stable").reset_index(drop=True)
    free()
    srow = cands["s_row"].to_numpy()
    edges = np.r_[np.searchsorted(srow, np.unique(srow)[::batch_s1]), len(srow)]
    parts = []
    for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        part = compute_pair_features(cands.iloc[lo:hi], s1, tgt, idf, n_jobs=n_jobs)
        part = add_s1_context(part, tgt)
        if predict_fn is not None:
            keep = {"s_row": part["s_row"].to_numpy(), "t_row": part["t_row"].to_numpy(),
                    "p": predict_fn(part).astype(np.float32)}
        else:
            keep = {"s_row": part["s_row"].to_numpy(), "t_row": part["t_row"].to_numpy(),
                    "X": part[ALL_FEATURES].to_numpy(np.float32)}
            keep.update(extra_fn(part) if extra_fn else {})
        parts.append(keep)
        del part
        free()
        log(f"  batch {b + 1}/{len(edges) - 1}: {hi - lo:,} pairs")
    del cands
    if not parts:
        return {"s_row": np.zeros(0, np.int32), "t_row": np.zeros(0, np.int32),
                "p": np.zeros(0, np.float32), "X": np.zeros((0, len(ALL_FEATURES)), np.float32)}
    out = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    del parts
    free()
    return out


def select_mask(s_row, t_row, p, thr, alpha, one_owner=True) -> np.ndarray:
    """Decision rule: p >= thr; each target kept only for its best S1 (the
    one-owner constraint verified in EDA cell 8); p >= alpha * best p of the
    S1.  Returns a boolean mask over the pairs."""
    mask = p >= thr
    idx = np.flatnonzero(mask)
    if one_owner and len(idx):
        d = pd.DataFrame({"t": t_row[idx], "p": p[idx], "i": idx}).sort_values(
            ["t", "p"], ascending=[True, False])
        best = d.drop_duplicates("t")["i"].to_numpy()
        mask = np.zeros(len(p), bool)
        mask[best] = True
        idx = best
    if alpha > 0 and len(idx):
        mx = pd.Series(p[idx]).groupby(s_row[idx]).transform("max").to_numpy()
        mask[idx[p[idx] < alpha * mx]] = False
    return mask


def fast_scores(s_row, y, mask, n_true_per_s, beta):
    """Micro and macro F-beta from aligned arrays (for the tuning grid).
    n_true_per_s[s] = number of true links of S1 row s (incl. those missed
    by blocking)."""
    n = len(n_true_per_s)
    tp_s = np.bincount(s_row[mask], weights=y[mask], minlength=n)
    np_s = np.bincount(s_row[mask], minlength=n).astype(float)
    tp, n_pred, n_true = tp_s.sum(), np_s.sum(), n_true_per_s.sum()
    p = tp / n_pred if n_pred else 1.0
    r = tp / n_true if n_true else 1.0
    with np.errstate(divide="ignore", invalid="ignore"):
        ps = np.where(np_s > 0, tp_s / np_s, np.where(n_true_per_s > 0, 0.0, 1.0))
        rs = np.where(n_true_per_s > 0, tp_s / n_true_per_s, np.where(np_s > 0, 0.0, 1.0))
        b2 = beta * beta
        fs = np.where((ps + rs) > 0, (1 + b2) * ps * rs / (b2 * ps + rs), 0.0)
    return {"micro_fbeta": fbeta(p, r, beta), "macro_fbeta": float(fs.mean())}


def doubtful_mask(d: dict, y: np.ndarray) -> np.ndarray:
    """Pairs whose label cannot be learnt from the evidence available.
    Removed ONLY from training folds - validation always scores everything.
      weak positive: true link with no name, address or number evidence
      twin negative: non-link indistinguishable from a link (same phonetic
                     name + same address) -> label noise / true duplicate"""
    weak_pos = ((y == 1) & (d["n_tset_skel"] < 35) & (d["n_idf_contain"] < 0.2)
                & (d["a_tset"] < 40) & (d["num_shared"] == 0) & (d["n_alt_best"] < 60))
    twin_neg = ((y == 0) & (d["n_exact_skel"] == 1) & (d["a_missing_t"] == 0)
                & ((d["a_tset"] >= 97) | ((d["num_jacc"] >= 0.999) & (d["num_s"] > 0))))
    return weak_pos | twin_neg


# ---------------------------------------------------------------------------
# train
# ---------------------------------------------------------------------------
def cmd_train(a):
    import lightgbm as lgb
    from sklearn.model_selection import GroupKFold

    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    n_jobs = n_workers(a.n_jobs)
    cfg = BlockConfig(top_k=a.top_k, top_k_name=a.top_k_name, top_k_addr=a.top_k_addr, n_jobs=n_jobs)
    data_dir = auto_data_dir(a.data_dir, "train")
    log(f"data dir: {data_dir} | out dir: {out} | workers: {n_jobs}")

    s1, tgt, gt = load_normalized(data_dir, "train", None if a.no_cache else out / "cache", n_jobs)
    truth = gt_pairs(gt)
    del gt
    rng = np.random.default_rng(42)
    if a.train_s1_sample and a.train_s1_sample < len(s1):
        keep_rows = np.sort(rng.choice(len(s1), a.train_s1_sample, replace=False)).astype(np.int32)
    else:
        keep_rows = np.arange(len(s1), dtype=np.int32)
    s1_ids = s1["entity_id"].to_numpy()
    sub_ids = s1_ids[keep_rows]
    log(f"S1 used for training/validation: {len(keep_rows):,} / {len(s1):,}")

    d = build_pairs(s1, tgt, cfg, n_jobs, keep_s_rows=keep_rows,
                    extra_fn=lambda part: {c: part[c].to_numpy(np.float32) for c in DOUBT_COLS})
    # labels
    t_ids = tgt["entity_id"].to_numpy()
    truth_sub = truth[truth["source1_entity_id"].isin(pd.Index(sub_ids))]
    pair_key = pd.MultiIndex.from_arrays([s1_ids[d["s_row"]], t_ids[d["t_row"]]])
    y = pair_key.isin(pd.MultiIndex.from_frame(truth_sub)).astype(np.int8)
    del pair_key
    recall_ceiling = y.sum() / max(len(truth_sub), 1)
    log(f"blocking recall (ceiling): {recall_ceiling:.4%} "
        f"({int(y.sum()):,} / {len(truth_sub):,} true links in candidates)")

    n_true_per_s = (truth_sub["source1_entity_id"].map(pd.Series(np.arange(len(s1)), index=s1_ids))
                    .value_counts().reindex(np.arange(len(s1)), fill_value=0).to_numpy().astype(float))
    eval_rows = np.zeros(len(s1), bool)
    eval_rows[keep_rows] = True

    # training-set cleaning (validation always sees everything)
    is_singleton = n_true_per_s[d["s_row"]] == 0
    doubt = doubtful_mask(d, y)
    train_ok = ~doubt
    if not a.keep_singletons:
        train_ok &= ~is_singleton
    log(f"train cleaning: doubtful pairs={int(doubt.sum()):,} "
        f"(pos {int((doubt & (y == 1)).sum()):,} / neg {int((doubt & (y == 0)).sum()):,}); "
        f"singleton-S1 pairs dropped={0 if a.keep_singletons else int(is_singleton.sum()):,}")
    for c in DOUBT_COLS:
        d.pop(c)
    free()

    X, s_row, t_row = d["X"], d["s_row"], d["t_row"]
    params = dict(LGB_PARAMS, num_threads=n_jobs)
    full = lgb.Dataset(X, y, feature_name=ALL_FEATURES, free_raw_data=False,
                       params={"max_bin": 255}).construct()
    oof = np.zeros(len(y), np.float32)
    best_iters = []
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=a.folds).split(s_row, y, s_row)):
        tr = tr[train_ok[tr]]
        bst = lgb.train(params, full.subset(np.sort(tr)), num_boost_round=a.rounds,
                        valid_sets=[full.subset(np.sort(va))],
                        callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = bst.predict(X[va], num_iteration=bst.best_iteration, num_threads=n_jobs)
        best_iters.append(bst.best_iteration or a.rounds)
        log(f"  fold {fold}: best_iter={best_iters[-1]}")
        del bst
        free()

    metric = f"{a.metric}_fbeta"
    ntps = np.where(eval_rows, n_true_per_s, 0.0)  # score only the sampled S1s
    best = None
    for thr in np.round(np.arange(0.10, 0.96, 0.025), 3):
        for alpha in (0.0, 0.2, 0.4, 0.6, 0.8):
            m = fast_scores(s_row, y, select_mask(s_row, t_row, oof, thr, alpha), ntps, a.beta)
            if best is None or m[metric] > best[2][metric]:
                best = (float(thr), float(alpha), m)
    thr, alpha, _ = best
    sel = select_mask(s_row, t_row, oof, thr, alpha)
    pred = pd.DataFrame({"source1_entity_id": s1_ids[s_row[sel]], "matched_id": t_ids[t_row[sel]]})
    m = evaluate(pred, truth_sub, sub_ids, a.beta)
    log(f"OOF best rule: thr={thr} alpha={alpha}")
    for k, v in m.items():
        log(f"  {k:16s} {v:.5f}" if isinstance(v, float) else f"  {k:16s} {v:,}")

    n_rounds = int(np.mean(best_iters) * 1.1) + 1
    log(f"final model on all clean rows ({int(train_ok.sum()):,}), rounds={n_rounds}")
    bst = lgb.train(params, full.subset(np.flatnonzero(train_ok)), num_boost_round=n_rounds)
    bst.save_model(str(out / "model.txt"))
    imp = pd.Series(bst.feature_importance("gain"), index=ALL_FEATURES).sort_values(ascending=False)
    imp.to_csv(out / "feature_importance.csv", header=["gain"])
    report = {"threshold": thr, "alpha": alpha, "beta": a.beta, "metric": metric,
              "oof": m, "blocking_recall": float(recall_ceiling),
              "block_cfg": {"top_k": a.top_k, "top_k_name": a.top_k_name, "top_k_addr": a.top_k_addr},
              "n_s1": int(len(keep_rows)), "n_pairs": int(len(y)),
              "n_doubtful": int(doubt.sum()), "features": ALL_FEATURES}
    (out / "config.json").write_text(json.dumps(report, indent=2, default=float))
    log(f"saved model + config to {out}")
    return report


# ---------------------------------------------------------------------------
# predict
# ---------------------------------------------------------------------------
def cmd_predict(a):
    import lightgbm as lgb

    out = Path(a.out_dir)
    n_jobs = n_workers(a.n_jobs)
    conf = json.loads((out / "config.json").read_text())
    bc = conf["block_cfg"]
    cfg = BlockConfig(top_k=bc["top_k"], top_k_name=bc["top_k_name"],
                      top_k_addr=bc["top_k_addr"], n_jobs=n_jobs)
    bst = lgb.Booster(model_file=str(out / "model.txt"))
    data_dir = auto_data_dir(a.data_dir, a.split)
    log(f"data dir: {data_dir} | split: {a.split}")

    s1, tgt, gt = load_normalized(data_dir, a.split, None if a.no_cache else out / "cache", n_jobs)
    feats = conf["features"]
    d = build_pairs(s1, tgt, cfg, n_jobs, predict_fn=lambda part: bst.predict(
        part[feats].to_numpy(np.float32), num_threads=n_jobs))
    sel = select_mask(d["s_row"], d["t_row"], d["p"], conf["threshold"], conf["alpha"])
    pairs = pd.DataFrame({"source1_entity_id": s1["entity_id"].to_numpy()[d["s_row"][sel]],
                          "matched_id": tgt["entity_id"].to_numpy()[d["t_row"][sel]]})

    agg = pairs.groupby("source1_entity_id")["matched_id"].agg(",".join)
    sub = pd.DataFrame({"source1_entity_id": s1["entity_id"]})
    sub["matched_entity_ids"] = sub["source1_entity_id"].map(agg).fillna("")
    path = Path(a.submission or out / f"submission_{a.split}.tsv")
    sub.to_csv(path, sep="\t", index=False)
    log(f"wrote {path} ({len(pairs):,} links for {len(sub):,} S1 records)")
    if is_kaggle() and a.split == "test" and str(data_dir).startswith("/kaggle/input"):
        shutil.copy(path, "/kaggle/working/submission.tsv")
        log("copied to /kaggle/working/submission.tsv")
    if gt is not None:
        m = evaluate(pairs, gt_pairs(gt), s1["entity_id"], conf["beta"])
        log("score on this split: " + json.dumps({k: round(v, 5) if isinstance(v, float) else v
                                                  for k, v in m.items()}))
        return m
    return None


def cmd_kaggle(a):
    cmd_train(a)
    free()
    a.split = "test"
    a.submission = None
    return cmd_predict(a)


def cmd_synth(a):
    from .synth import write_synthetic
    write_synthetic(a.out_dir, n_train=a.n_train, n_test=a.n_test, seed=a.seed)


def _add_train_args(t):
    t.add_argument("--train-s1-sample", type=int, default=250_000,
                   help="S1 records used for training/validation (0 = all). "
                        "250k fits the 30 GB Kaggle RAM limit comfortably")
    t.add_argument("--folds", type=int, default=5)
    t.add_argument("--rounds", type=int, default=3000)
    t.add_argument("--beta", type=float, default=0.5)
    t.add_argument("--metric", choices=["micro", "macro"], default="micro")
    t.add_argument("--keep-singletons", action="store_true",
                   help="keep S1 records without any true match in the TRAINING folds")
    t.add_argument("--top-k", type=int, default=40)
    t.add_argument("--top-k-name", type=int, default=10)
    t.add_argument("--top-k-addr", type=int, default=10)


def main(argv=None):
    ap = argparse.ArgumentParser(prog="entity_matching")
    sub = ap.add_subparsers(dest="cmd", required=True)
    for name in ("train", "predict", "kaggle"):
        p = sub.add_parser(name)
        p.add_argument("--data-dir", default=None, help="auto-detected when omitted")
        p.add_argument("--out-dir", default=str(default_out_dir()))
        p.add_argument("--n-jobs", type=int, default=None, help="default: all available cores")
        p.add_argument("--no-cache", action="store_true",
                       help="do not cache normalised data in OUT/cache")
    _add_train_args(sub.choices["train"])
    _add_train_args(sub.choices["kaggle"])
    for name in ("predict", "kaggle"):
        sub.choices[name].add_argument("--split", default="test")
        sub.choices[name].add_argument("--submission", default=None)
    s = sub.add_parser("synth")
    s.add_argument("--out-dir", required=True)
    s.add_argument("--n-train", type=int, default=6000)
    s.add_argument("--n-test", type=int, default=3000)
    s.add_argument("--seed", type=int, default=0)
    a, _ = ap.parse_known_args(argv)  # tolerate Jupyter's -f kernel.json
    {"train": cmd_train, "predict": cmd_predict, "kaggle": cmd_kaggle, "synth": cmd_synth}[a.cmd](a)


if __name__ == "__main__":
    main()

Writing entity_matching/run.py


## Step 2 — sanity check of the hardcoded patterns
These pairs are **real true matches taken from the EDA notebook**. After normalisation both sides should give the
same phonetic skeleton, whatever the script, OCR noise, brackets or legal-form position.

In [12]:
import importlib, entity_matching
for m in ["patterns", "normalize", "utils", "data", "blocking", "features", "metrics", "synth", "run"]:
    importlib.reload(importlib.import_module(f"entity_matching.{m}"))

import pandas as pd
from entity_matching.normalize import normalize_name, normalize_address

pairs = [
    ("Gold Solutions", "গোল্ড সলিউশনস"),
    ("Jain Builders Private Limited", "जैन बिल्डर्स प्राइवेट लिमिटेड"),
    ("Creative Builders", "ಕ್ರಿಯೇಟಿವ್ ಬಿಲ್ಡರ್ಸ್"),
    ("Arihant Technology Limited", "Arihant टेक्नोलॉजी लिमिटेड"),
    ("Creative Intermediate LLC", "Llc Creative Intermediate"),
    ("Bright Seafood Inc", "Bright Séafood Inc"),
    ("South Academy", "5outh Academy"),
    ("Washington, Harris and Burrell", "WASHINGT0N, HARRIS AND BURRELL"),
    ("Midwest Applied, LLC", "Midwest Applied, ([LLC])"),
    ("Fort Worth Telecommunication", "Fort Worth Telecommunication L.L.C. (ID: 49166)"),
    ("Summit Ministries", "Summit Ministries lnc"),
    ("Bright Automation Enterprises", "Bright Bright Automation Enterprises"),
    ("Integrity Services (India) Limited", "*** Integrity Services (India)  Limited"),
]
rows = []
for a, b in pairs:
    na, nb = normalize_name(a), normalize_name(b)
    rows.append({"S1 name": a, "S2/S3 name": b, "S1 skeleton": na["skel"], "S2/S3 skeleton": nb["skel"],
                 "legal": f"{na['legal']} / {nb['legal']}", "match": na["skel"] == nb["skel"]})
df_names = pd.DataFrame(rows)
print(f"{df_names['match'].sum()} / {len(df_names)} real true pairs reduced to the same skeleton")
df_names

13 / 13 real true pairs reduced to the same skeleton


,S1 name,S2/S3 name,S1 skeleton,S2/S3 skeleton,legal,match
0,Gold Solutions,গোল্ড সলিউশনস,klt slsns,klt slsns,/,True
1,Jain Builders Private Limited,जैन बिल्डर्स प्राइवेट लिमिटेड,jn pltrs,jn pltrs,ltd pvt / ltd pvt,True
2,Creative Builders,ಕ್ರಿಯೇಟಿವ್ ಬಿಲ್ಡರ್ಸ್,krtv pltrs,krtv pltrs,/,True
3,Arihant Technology Limited,Arihant टेक्नोलॉजी लिमिटेड,arnt tknlj,arnt tknlj,ltd / ltd,True
4,Creative Intermediate LLC,Llc Creative Intermediate,krtv intrnt,krtv intrnt,llc / llc,True
5,Bright Seafood Inc,Bright Séafood Inc,prkt sft,prkt sft,inc / inc,True
6,South Academy,5outh Academy,st aktn,st aktn,/,True
7,"Washington, Harris and Burrell","WASHINGT0N, HARRIS AND BURRELL",vsnktn hrs prl,vsnktn hrs prl,/,True
8,"Midwest Applied, LLC","Midwest Applied, ([LLC])",ntvst aplt,ntvst aplt,llc / llc,True
9,Fort Worth Telecommunication,Fort Worth Telecommunication L.L.C. (ID: 49166),frt vrt tlknksn,frt vrt tlknksn,/ llc,True


In [13]:
addr = [
    ("##16978 Moore Rd, <NULL>, Andalusia, Alabama", "US"),
    ("16978 MOORE ROAD, ANDALUSIA, AL", "US"),
    ("PLOT NO B-78/1, ADDITIONAL MIDC ANAND NAGAR, AMBERNATH EAST, THANE, महाराष्ट्र", "India"),
    ("Thane, MH, Plot No. B-78/1, Additional Midc Anand Nagar, Ambernath East", "India"),
    ("05131 COPPER MEADOW LN, WEST JORDAN CITY, UT", "US"),
    ("H.No1-8-67/P Apiic Kamalanagar, Kushaiguda, Hyderabad, Rangareddy, Telangana", "India"),
    ("20 bis RUE jules lefebvre, Lille, Hauts-de-France", "France"),
]
pd.DataFrame([{"raw address": a, "country": c, **{k: v for k, v in normalize_address(a, c).items()
               if k in ("alpha", "nums", "numstreet", "state")}} for a, c in addr])

,raw address,country,alpha,nums,numstreet,state
0,"##16978 Moore Rd, <NULL>, Andalusia, Alabama",US,moore rd andalusia,16978,16978_moore,al
1,"16978 MOORE ROAD, ANDALUSIA, AL",US,moore rd andalusia,16978,16978_moore,al
2,"PLOT NO B-78/1, ADDITIONAL MIDC ANAND NAGAR, A...",India,additional midc anand nagar ambernath e thane,b-78/1,,mh
3,"Thane, MH, Plot No. B-78/1, Additional Midc An...",India,thane additional midc anand nagar ambernath e,b-78/1,,mh
4,"05131 COPPER MEADOW LN, WEST JORDAN CITY, UT",US,copper meadow ln w jordan,5131,5131_copper,ut
5,"H.No1-8-67/P Apiic Kamalanagar, Kushaiguda, Hy...",India,apiic kamalanagar kushaiguda hyderabad rangareddy,1-8-67/p,1-8-67/p_apiic,tg
6,"20 bis RUE jules lefebvre, Lille, Hauts-de-France",France,bis rue jules lefebvre lille,20,20_bis,hdf


## Step 3 — synthetic demo (reproduces the 3k-S1 test result including France)
This generates data in the exact competition layout, with every noise type found in the EDA:
- Indic scripts; legal-form swaps, brackets and moves; OCR digits; accents
- `***` / `(ID: …)` junk; domains; DBA aliases; repeated or shuffled words
- `NULL` placeholders, shuffled address components, abbreviations, missing addresses
- hard-negative generic names and singletons

Train has US + India. **Test adds the unseen France**, as in the competition.

Last run: 3k-S1 test including France → **precision 0.997, recall 0.957, F0.5 0.989**.
These are synthetic scores. They check correctness, **not** the leaderboard. Set `RUN_DEMO = False` to skip.

In [14]:
RUN_DEMO = True

if RUN_DEMO:
    from entity_matching.run import main
    main(["synth", "--out-dir", "synth_demo", "--n-train", "6000", "--n-test", "3000", "--seed", "0"])
    main(["kaggle", "--data-dir", "synth_demo", "--out-dir", "em_demo",
          "--train-s1-sample", "0", "--no-cache"])

train: S1=6,000 S2=13,722 S3=13,561 links=20,987


test: S1=3,000 S2=6,650 S3=6,899 links=10,423
[    0.8s |   0.2 GB] data dir: synth_demo | out dir: em_demo | workers: 4


[    0.8s |   0.2 GB] loading train from synth_demo


[    1.0s |   0.2 GB]   S1=6,000 targets=27,283


[    1.0s |   0.2 GB] normalising S1 ...


...

[   32.5s |   0.4 GB] building IDF tables from targets ...


[   32.6s |   0.4 GB] pair features ...


[   33.9s |   0.4 GB]   batch 1/1: 79,209 pairs


[   34.0s |   0.4 GB] wrote em_demo/submission_test.tsv (9,999 links for 3,000 S1 records)


[   34.1s |   0.4 GB] score on this split: {"micro_precision": 0.9974, "micro_recall": 0.95683, "micro_fbeta": 0.98901, "macro_precision": 0.99404, "macro_recall": 0.95862, "macro_fbeta": 0.98383, "tp": 9973, "n_pred": 9999, "n_true": 10423}


In [15]:
if RUN_DEMO:
    import json
    from entity_matching.data import gt_pairs, load_split
    from entity_matching.metrics import evaluate
    conf = json.load(open("em_demo/config.json"))
    sub = pd.read_csv("em_demo/submission_test.tsv", sep="\t", dtype=str, keep_default_na=False)
    s1_test, _, gt_test = load_split("synth_demo", "test")
    res = evaluate(gt_pairs(sub), gt_pairs(gt_test), s1_test["entity_id"], beta=0.5)
    summary = pd.DataFrame([
        {"run": "6k S1 train (OOF)", "blocking recall": conf["blocking_recall"],
         "precision": conf["oof"]["micro_precision"], "recall": conf["oof"]["micro_recall"],
         "F0.5 micro": conf["oof"]["micro_fbeta"], "F0.5 macro": conf["oof"]["macro_fbeta"]},
        {"run": "3k S1 test incl. France", "blocking recall": None,
         "precision": res["micro_precision"], "recall": res["micro_recall"],
         "F0.5 micro": res["micro_fbeta"], "F0.5 macro": res["macro_fbeta"]},
    ])
    # per-country breakdown of the test result
    by_c = []
    ctry = s1_test.set_index("entity_id")["country"].astype(str)
    for c in sorted(ctry.unique()):
        ids = ctry.index[ctry == c]
        m = evaluate(gt_pairs(sub), gt_pairs(gt_test), ids, beta=0.5)
        by_c.append({"country": c, "S1": len(ids), "precision": m["micro_precision"],
                     "recall": m["micro_recall"], "F0.5": m["micro_fbeta"]})
    display(summary.round(4))
    display(pd.DataFrame(by_c).round(4))

,run,blocking recall,precision,recall,F0.5 micro,F0.5 macro
0,6k S1 train (OOF),0.9702,0.9971,0.9521,0.9878,0.9821
1,3k S1 test incl. France,NaN,0.9974,0.9568,0.9890,0.9838


,country,S1,precision,recall,F0.5
0,France,458,0.9948,0.9505,0.9856
1,India,1347,0.9973,0.9521,0.9880
2,US,1195,0.9985,0.9648,0.9915


## Step 4 — real competition data
This runs automatically when the dataset is attached: the code searches `/kaggle/input/**` for `train_source1.tsv`.
To point at a specific folder, set `DATA_DIR`.

- Training uses `TRAIN_S1_SAMPLE` S1 records (250k fits Kaggle's 30 GB RAM). Blocking and the
  target-competition features still use **all** S1 records.
- The log prints elapsed time and RAM for every step, plus the **blocking recall ceiling**, which is the best
  recall any model can reach, and the OOF precision / recall / F0.5.
- Normalised data is cached in `em_model/cache/`, so a re-run skips normalisation.

Expected runtime on a 4-core Kaggle CPU notebook, extrapolated from the synthetic scale test: roughly 30–60 min.

In [16]:
from pathlib import Path

DATA_DIR = None            # e.g. "/kaggle/input/amazon-ml-challenge-2026/dataset"; None = auto-detect
TRAIN_S1_SAMPLE = 250_000  # 0 = all S1 records (needs more RAM)
OUT_DIR = "/kaggle/working/em_model" if Path("/kaggle/working").is_dir() else "em_model"

def find_real_data():
    if DATA_DIR:
        return DATA_DIR
    root = Path("/kaggle/input")
    hits = sorted(root.rglob("train_source1.tsv")) if root.is_dir() else []
    if not hits:
        return None
    d = hits[0].parent
    return str(d.parent if d.name == "train" else d)

real_dir = find_real_data()
if real_dir is None:
    print("Competition dataset not found under /kaggle/input - attach it (or set DATA_DIR) and re-run this cell.")
else:
    print("dataset:", real_dir)
    from entity_matching.run import main
    main(["kaggle", "--data-dir", real_dir, "--out-dir", OUT_DIR,
          "--train-s1-sample", str(TRAIN_S1_SAMPLE)])

Competition dataset not found under /kaggle/input - attach it (or set DATA_DIR) and re-run this cell.


## Step 5 — inspect results

In [17]:
import json
cfg_path = Path(OUT_DIR) / "config.json"
if cfg_path.exists():
    conf = json.loads(cfg_path.read_text())
    print(f"decision rule: p >= {conf['threshold']}  alpha = {conf['alpha']}  (tuned for {conf['metric']})")
    print(f"blocking recall ceiling: {conf['blocking_recall']:.4%}")
    display(pd.Series(conf["oof"]).to_frame("OOF").T)
    display(pd.read_csv(Path(OUT_DIR) / "feature_importance.csv", index_col=0).head(20))
    sub_path = Path("/kaggle/working/submission.tsv")
    if sub_path.exists():
        sub = pd.read_csv(sub_path, sep="\t", dtype=str, keep_default_na=False)
        n_links = sub["matched_entity_ids"].str.count(",").add(1).where(sub["matched_entity_ids"] != "", 0)
        print(f"submission: {len(sub):,} S1 rows, {int(n_links.sum()):,} links, "
              f"{(n_links == 0).mean():.2%} S1 with no match")
        display(sub.head(10))
else:
    print("No real-data model yet - run Step 4 with the dataset attached.")

No real-data model yet - run Step 4 with the dataset attached.
